# EGCB: entropy-guided circuit breakers for LLM-augmented intrusion detection

This notebook replaces the earlier experiment cells. **Every table and figure in the revised paper should come from this notebook only.** All numbers are measured; nothing is scaled, typed in, or copied between runs.

**Pipeline**
1. Load CIC-IDS2017, CICIoT2023 and NSL-KDD with correct columns and binary labels (no label columns inside the features).
2. Train the **reflex kernel** (Random Forest) on the training split. Measure accuracy, F1, false-negative rate and per-flow latency on the test split.
3. Route test flows with confidence `u(x) = |P(attack|x) - 0.5| <= DELTA` to the LLM. Also build a balanced random "stress set" of test flows per dataset, so the circuit breaker is tested on enough samples. (On the stress set the reflex is usually confident, so a tripped flow falls back to a strong prediction. Read the stress results through "LLM accuracy when it answered" and the AUROC table, not system accuracy alone.)
4. **Prompt**: with `PROMPT_MODE = "retrieval"` the LLM also sees the most similar labelled benign and attack flows from the training split (retrieval-augmented reasoning); `"zero_shot"` shows only the flow. Each mode saves to its own folder, so both can be reported.
5. **Naive run**: each LLM (small / medium / large, one model family) answers every evaluation flow with greedy decoding and no breaker. Per-token entropy and per-token timestamps are logged.
5. Choose the entropy threshold `tau` on **training flows that the cascade would route** (identified by out-of-bag Random Forest confidence), using the Eq. 6 objective. No test flow influences `tau`.
6. **Live EGCB run** on the test split with the chosen `tau` (the circuit breaker really stops generation). Report measured tokens, latency and accuracy against the naive run.
7. Ablations and baselines: reflex-only, LLM naive, EGCB, EGCB without smoothing (k=1), a fixed token budget with the same compute, and a random breaker with the same trip rate.
8. Test whether entropy actually predicts LLM errors (AUROC), and compute end-to-end cascade results.

**How to run.** Upload the three zip files to `/content`. Run everything once with `QUICK_TEST = True` (a few minutes). Then set `QUICK_TEST = False` and run again. Results are saved after every sample, so if Colab disconnects, re-running continues where it stopped.

**Hardware.** A GPU with ≥ 24 GB (L4 or A100) runs all three models in bf16. On a 16 GB T4, the 7B model is loaded in 4-bit automatically, and this is recorded in the output so you can report it.

In [ ]:
# =====================================================================
# 0. CONFIGURATION  (the only cell you should need to edit)
# =====================================================================
QUICK_TEST = True        # True: tiny end-to-end check. False: full experiment for the paper.

SEED = 42

ZIP_PATHS = {
    "CIC-IDS2017": "/content/CIC 2017.zip",
    "CICIoT2023":  "/content/CICIoT2023.zip",
    "NSL-KDD":     "/content/KDD.zip",
}
DATA_ROOT   = "/content/datasets"
MOUNT_DRIVE = True       # save results to Google Drive so a disconnect does not lose them
RESULTS_DIR = "/content/drive/MyDrive/egcb_results" if MOUNT_DRIVE else "/content/egcb_results"

# ---- data sampling (keeps memory manageable on Colab) ----
CIC_BENIGN_CAP       = 200_000   # max benign flows kept from CIC-IDS2017 (after de-duplication)
CIC_ATTACK_CLASS_CAP = 20_000    # max flows kept per attack type
CIC_TEST_SIZE        = 0.30      # CIC-IDS2017 has no official split -> stratified 70/30
IOT_TRAIN_ROWS       = 200_000   # random rows drawn from the official CICIoT2023 train.csv
IOT_TEST_ROWS        = 60_000    # random rows drawn from the official CICIoT2023 test.csv
# NSL-KDD uses the official KDDTrain+ / KDDTest+ files in full.

# ---- reflex kernel ----
RF_PARAMS = dict(n_estimators=100, class_weight="balanced_subsample", oob_score=True,
                 n_jobs=-1, random_state=SEED)   # OOB probabilities give honest 'would-be-routed' training flows
DELTA       = 0.20                       # accept reflex prediction if u(x) > DELTA (Eq. 5)
DELTA_SWEEP = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]
FEATURE_TOP_K = 20                       # phi(x): the K most important RF features are written into the prompt

# ---- prompt mode ----
# "retrieval": the prompt also shows the N_REF_PER_CLASS most similar BENIGN and ATTACK flows
#              from the TRAINING split, with their labels (retrieval-augmented reasoning)
# "zero_shot": only the flow's own feature values
PROMPT_MODE        = "retrieval"
N_REF_PER_CLASS    = 2
REF_POOL_PER_CLASS = 50_000   # training flows per class searched for references

# ---- LLM evaluation sets ----
N_STRESS   = 200    # balanced random test flows per dataset (half benign, half attack), not routed
ROUTED_CAP = 200    # max routed (uncertain) test flows per dataset sent to the LLM
N_CALIB    = 150    # TRAINING flows per dataset used ONLY to choose tau: flows whose out-of-bag
                    # reflex confidence is <= DELTA (i.e. flows the cascade would route). Never test data.

# ---- LLMs: one family, three sizes, so "small < medium < large" is actually true ----
MODELS = {
    "small":  {"hf_id": "Qwen/Qwen2.5-0.5B-Instruct", "params_b": 0.49},
    "medium": {"hf_id": "Qwen/Qwen2.5-3B-Instruct",   "params_b": 3.09},
    "large":  {"hf_id": "Qwen/Qwen2.5-7B-Instruct",   "params_b": 7.62},
}
FORCE_4BIT = False  # if False, 4-bit is used only when a model does not fit in GPU memory

# ---- circuit breaker ----
MAX_NEW_TOKENS = 256   # T_max (Eq. 13), in generated tokens (room for 2-4 sentences of reasoning)
WINDOW         = 8     # k, smoothing window of Eq. 10 (tokens)
MIN_TOKENS     = 8     # the breaker may not trip before this many tokens have been generated
LAMBDA         = 0.7   # Eq. 6: J = LAMBDA * error + (1 - LAMBDA) * (tokens / MAX_NEW_TOKENS)
TAU_FIXED      = None  # set a number to skip calibration and use a fixed tau for every model
RUN_LIVE       = True  # run the real EGCB (breaker active) on the test split

BOOTSTRAP           = 1000  # bootstrap resamples for 95% confidence intervals
RANDOM_TRIP_REPEATS = 200   # repeats of the random-breaker baseline

if QUICK_TEST:
    CIC_BENIGN_CAP, CIC_ATTACK_CLASS_CAP = 30_000, 3_000
    IOT_TRAIN_ROWS, IOT_TEST_ROWS = 30_000, 10_000
    N_STRESS, ROUTED_CAP, N_CALIB = 12, 6, 8
    MODELS = {"small": MODELS["small"], "medium": MODELS["medium"]}   # check behaviour on 2 sizes
    MAX_NEW_TOKENS = 192
    BOOTSTRAP, RANDOM_TRIP_REPEATS = 200, 50
    RESULTS_DIR = RESULTS_DIR + "_quicktest"

RESULTS_DIR = RESULTS_DIR + f"_{PROMPT_MODE}"     # each prompt mode gets its own results folder
print("QUICK_TEST =", QUICK_TEST, "| PROMPT_MODE =", PROMPT_MODE, "| results ->", RESULTS_DIR)

In [ ]:
# =====================================================================
# 1. INSTALL (Colab already has torch, pandas, scikit-learn, matplotlib)
# =====================================================================
!pip -q install "transformers>=4.44,<5" "accelerate>=0.33" bitsandbytes

In [ ]:
# =====================================================================
# 2. IMPORTS, SEEDS, OUTPUT FOLDERS
# =====================================================================
import os, re, gc, glob, json, time, math, zipfile, random, platform, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(SEED); np.random.seed(SEED)

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted (", e, ") -> saving locally")
        RESULTS_DIR = RESULTS_DIR.replace("/content/drive/MyDrive", "/content")

for sub in ["tables", "figures", "llm_runs", "data"]:
    os.makedirs(os.path.join(RESULTS_DIR, sub), exist_ok=True)

def out(*parts):
    return os.path.join(RESULTS_DIR, *parts)

def save_table(df, name):
    df.to_csv(out("tables", name + ".csv"), index=False)
    print(f"[saved] tables/{name}.csv")

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(out("figures", f"{name}.{ext}"), dpi=300, bbox_inches="tight")
    print(f"[saved] figures/{name}.png/.pdf")

CONFIG = {k: v for k, v in globals().items()
          if k.isupper() and isinstance(v, (int, float, str, bool, dict, list, type(None)))}
with open(out("config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
print("Config saved to", out("config.json"))

In [ ]:
# =====================================================================
# 3. USE EXTRACTED DATASET FOLDERS FROM GOOGLE DRIVE (optional, safe to run always)
#     If the zips are broken or you have the extracted folders in Drive, this uses them.
#     It finds the extracted files in your Drive, copies them to local disk
#     in the layout cell 4 expects, and marks them as ready, so cell 4 skips
#     the zip for those datasets. Datasets it cannot find still use the zip.
# =====================================================================
import os, shutil

DRIVE_ROOT = "/content/drive/MyDrive"

# If auto-detection picks the wrong folder, put the folder path here, e.g.
# MANUAL_FOLDERS = {"CICIoT2023": "/content/drive/MyDrive/CICIoT2023"}
MANUAL_FOLDERS = {}

WANTED = {   # dataset -> files cell 4 needs
    "CICIoT2023": ["train.csv", "test.csv"],
    "NSL-KDD":    ["KDDTrain+.txt", "KDDTest+.txt"],
    "CIC-IDS2017": ["*pcap_ISCX.csv"],
}
HINTS = {"CICIoT2023": "iot", "NSL-KDD": "kdd", "CIC-IDS2017": "cic"}

def scan(root, max_depth=6):
    files = []
    base = root.rstrip(os.sep).count(os.sep)
    for r, dirs, fs in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__MACOSX"]
        if r.count(os.sep) - base >= max_depth:
            dirs[:] = []
        files += [os.path.join(r, f) for f in fs if not f.startswith(".")]
    return files

def match(name, files):
    """Return {needed_name: path} for one dataset, or None if not all are present."""
    found = {}
    if name == "CIC-IDS2017":
        hits = [f for f in files if f.endswith("pcap_ISCX.csv")]
        return {os.path.basename(h): h for h in hits} if hits else None
    for want in WANTED[name]:
        cands = [f for f in files if os.path.basename(f).lower() == want.lower()]
        if len(cands) > 1:   # prefer paths that mention the dataset
            cands = sorted(cands, key=lambda p: (HINTS[name] not in p.lower(), len(p)))
        if not cands:
            return None
        found[want] = cands[0]
    return found

def install_from_drive():
    if not os.path.isdir(DRIVE_ROOT):
        print("Google Drive is not mounted - run cell 2 first (MOUNT_DRIVE = True).")
        return
    for name in WANTED:
        dest = os.path.join(DATA_ROOT, name)
        if os.path.exists(os.path.join(dest, ".extracted_ok")):
            print(f"{name}: already ready in {dest}")
            continue
        search_root = MANUAL_FOLDERS.get(name, DRIVE_ROOT)
        print(f"{name}: searching {search_root} ...")
        hits = match(name, scan(search_root))
        if not hits:
            print(f"   not found in Drive -> cell 4 will use the zip instead")
            continue
        if os.path.isdir(dest):
            shutil.rmtree(dest)
        for want, src in hits.items():
            if name == "CICIoT2023":
                sub = "train" if want.lower().startswith("train") else "test"
                target = os.path.join(dest, "CICIOT23", sub, want)
            else:
                target = os.path.join(dest, os.path.basename(src))
            os.makedirs(os.path.dirname(target), exist_ok=True)
            size = os.path.getsize(src) / 1e9
            print(f"   copying {src}  ({size:.2f} GB)")
            shutil.copyfile(src, target)
            if os.path.getsize(target) != os.path.getsize(src):
                raise RuntimeError(f"Copy of {src} is incomplete - re-run this cell")
        open(os.path.join(dest, ".extracted_ok"), "w").close()
        print(f"{name}: ready ({len(hits)} file(s)) in {dest}")

install_from_drive()

In [ ]:
# =====================================================================
# 4. LOAD ALL DATASETS  ->  Table 1 (dataset summary)
#    Self-contained: redefines unzip + loaders, so it does not depend on
#    older versions of cell 3 still living in memory.
# =====================================================================
import os, gc, glob, json, shutil, zipfile
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------
# 4a. Find the zip files (checks /content, Google Drive and ZIP_PATHS)
# ---------------------------------------------------------------------
ZIP_NAMES = {"CIC-IDS2017": "CIC 2017.zip", "CICIoT2023": "CICIoT2023.zip", "NSL-KDD": "KDD.zip"}
SEARCH_DIRS = ["/content", "/content/drive/MyDrive", "/content/drive/MyDrive/datasets"]

def locate_zip(name):
    cands = [ZIP_PATHS.get(name)] + [os.path.join(d, ZIP_NAMES[name]) for d in SEARCH_DIRS]
    for p in cands:
        if p and os.path.isfile(p):
            return p
    return None

def extracted_ok(dest):
    return os.path.exists(os.path.join(dest, ".extracted_ok"))

def needed(name, member):
    """Which archive members the pipeline actually uses."""
    m = member.lower()
    if member.startswith("__MACOSX") or os.path.basename(member).startswith(".") or member.endswith("/"):
        return False
    if name == "CICIoT2023":                       # validation.csv is never used
        return m.endswith(".csv") and ("train" in m or "test" in m)
    if name == "NSL-KDD":
        return os.path.basename(member) in ("KDDTrain+.txt", "KDDTest+.txt")
    return m.endswith(".csv")

def extract_member_fallback(zp, member, dest):
    """Try the system tools, which are more tolerant than Python's zipfile."""
    import subprocess
    for cmd in (["unzip", "-o", "-q", zp, member, "-d", dest],
                ["7z", "x", "-y", f"-o{dest}", zp, member]):
        if shutil.which(cmd[0]) is None:
            continue
        r = subprocess.run(cmd, capture_output=True, text=True)
        target = os.path.join(dest, member)
        if r.returncode == 0 and os.path.isfile(target) and os.path.getsize(target) > 0:
            return cmd[0]
    return None

def unzip_all():
    for name in ZIP_NAMES:
        dest = os.path.join(DATA_ROOT, name)
        if extracted_ok(dest):
            print(f"{name}: already extracted (verified)")
            continue
        zp = locate_zip(name)
        if zp is None:
            raise FileNotFoundError(
                f"{name}: zip not found. Looked for {[ZIP_PATHS.get(name)] + [os.path.join(d, ZIP_NAMES[name]) for d in SEARCH_DIRS]}")
        size_gb = os.path.getsize(zp) / 1e9
        print(f"{name}: opening {zp} ({size_gb:.2f} GB)")
        if not zipfile.is_zipfile(zp):
            raise RuntimeError(f"{name}: {zp} is not a valid zip (upload incomplete?). Re-upload it.")
        if os.path.isdir(dest):
            shutil.rmtree(dest)
        os.makedirs(dest, exist_ok=True)
        failed = []
        with zipfile.ZipFile(zp) as z:
            members = [i for i in z.infolist() if needed(name, i.filename)]
            if not members:
                raise RuntimeError(f"{name}: none of the expected files are in {zp}. Contents: {z.namelist()[:15]}")
            for info in members:
                try:
                    z.extract(info, dest)
                    print(f"   ok      {info.filename}  ({info.file_size/1e9:.2f} GB)")
                except Exception as e:
                    part = os.path.join(dest, info.filename)
                    if os.path.exists(part):
                        os.remove(part)
                    tool = extract_member_fallback(zp, info.filename, dest)
                    if tool:
                        print(f"   ok      {info.filename}  (Python failed: {e}; extracted with {tool})")
                    else:
                        failed.append((info.filename, f"{type(e).__name__}: {e}"))
                        print(f"   FAILED  {info.filename}: {type(e).__name__}: {e}")
        if failed:
            raise RuntimeError(
                f"{name}: {len(failed)} file(s) inside {os.path.basename(zp)} are damaged and could not be "
                f"extracted by Python, unzip or 7z: {[f for f, _ in failed]}.\n"
                f"The zip on Colab is corrupted (usually a browser upload problem). Fix: re-create the zip on your "
                f"computer, upload it to Google Drive (not the Colab Files panel), wait for the upload to finish, "
                f"and set ZIP_PATHS['{name}'] to the Drive path. Or upload the extracted CSV files directly: put "
                f"train.csv and test.csv under {dest}/ and create an empty file {dest}/.extracted_ok")
        open(os.path.join(dest, ".extracted_ok"), "w").close()
        print(f"{name}: extracted {len(members)} file(s) to {dest}")

# ---------------------------------------------------------------------
# 4b. Loaders
# ---------------------------------------------------------------------
KDD_FEATURES = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
]
KDD_CAT = ["protocol_type", "service", "flag"]
LEAK_COLS = {"flow id", "source ip", "src ip", "destination ip", "dst ip", "timestamp"}

def find_files(root, pattern):
    hits = [p for p in glob.glob(os.path.join(root, "**", pattern), recursive=True)
            if "__MACOSX" not in p and not os.path.basename(p).startswith(".")]
    return sorted(hits, key=lambda p: (p.count(os.sep), p))

def label_column(df, path):
    for c in df.columns:
        if str(c).strip().lower() in ("label", "class", "attack", "attack_type"):
            return c
    raise KeyError(f"No label column in {path}. Columns end with: {list(df.columns)[-5:]}")

def to_float_frame(df):
    num = df.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    ok = num.notna().all(axis=1).values
    return num[ok].astype(np.float32), ok

def drop_useless_columns(train, feat_cols, cat_cols):
    dropped, keep, seen = [], [], {}
    for c in feat_cols:
        if c not in cat_cols and train[c].nunique(dropna=False) <= 1:
            dropped.append((c, "constant")); continue
        h = int(pd.util.hash_pandas_object(train[c], index=False).sum())
        if h in seen and train[c].equals(train[seen[h]]):
            dropped.append((c, f"duplicate of {seen[h]}")); continue
        seen[h] = c; keep.append(c)
    return keep, dropped

def load_cic(root):
    files = find_files(root, "*.csv")
    if not files:
        raise FileNotFoundError(f"No CSV files under {root}")
    parts, n_raw = [], 0
    for f in files:
        df = pd.read_csv(f, low_memory=False, encoding_errors="replace")
        df.columns = [str(c).strip() for c in df.columns]
        df = df.loc[:, ~pd.Index(df.columns).duplicated()]
        n_raw += len(df)
        lab_col = label_column(df, f)
        lab = df.pop(lab_col).astype(str).str.strip()
        df = df[[c for c in df.columns if c.lower() not in LEAK_COLS]]
        num, ok = to_float_frame(df)
        num["attack_type"] = lab.values[ok]
        num = num.drop_duplicates()
        # cap benign per file early to save memory (final caps applied below)
        ben = num["attack_type"] == "BENIGN"
        if ben.sum() > CIC_BENIGN_CAP:
            num = pd.concat([num[~ben], num[ben].sample(CIC_BENIGN_CAP, random_state=SEED)])
        parts.append(num)
        print(f"  {os.path.basename(f)}: {len(num):,} clean unique rows kept")
        del df; gc.collect()
    cols = parts[0].columns
    for f, p in zip(files, parts):
        if list(p.columns) != list(cols):
            raise ValueError(f"{f} has different columns from {files[0]}")
    df = pd.concat(parts, ignore_index=True); del parts; gc.collect()
    n_clean = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    groups = []
    for cls, g in df.groupby("attack_type"):
        cap = CIC_BENIGN_CAP if cls == "BENIGN" else CIC_ATTACK_CLASS_CAP
        groups.append(g.sample(n=min(len(g), cap), random_state=SEED))
    df = pd.concat(groups).sample(frac=1, random_state=SEED).reset_index(drop=True)
    df["y"] = (df["attack_type"] != "BENIGN").astype(int)
    counts = df["attack_type"].value_counts()
    strat = df["attack_type"].where(df["attack_type"].map(counts) >= 2, "RARE")
    train, test = train_test_split(df, test_size=CIC_TEST_SIZE, stratify=strat, random_state=SEED)
    stats = dict(raw_rows=n_raw, clean_unique_rows=n_clean, kept_rows=len(df),
                 split="stratified 70/30 (no official split)", files=len(files),
                 attack_types=counts.to_dict())
    return train, test, [], stats

def count_rows(path):
    with open(path, "rb") as f:
        return sum(1 for _ in f) - 1

def sample_csv(path, target, seed):
    total = count_rows(path)
    frac = min(1.0, target / max(total, 1))
    parts = [ch.sample(frac=frac, random_state=seed + i)
             for i, ch in enumerate(pd.read_csv(path, chunksize=500_000, low_memory=False))]
    return pd.concat(parts, ignore_index=True), total

def prep_iot(df, path):
    df.columns = [str(c).strip() for c in df.columns]
    lab = df.pop(label_column(df, path)).astype(str).str.strip()
    num, ok = to_float_frame(df)
    num["attack_type"] = lab.values[ok]
    num["y"] = (~num["attack_type"].str.lower().str.startswith("benign")).astype(int)
    return num.reset_index(drop=True)

def load_iot(root):
    csvs = find_files(root, "*.csv")
    def pick(word):
        hits = [p for p in csvs if os.path.basename(p).lower().startswith(word)
                or f"{os.sep}{word}{os.sep}" in p.lower()]
        return hits[0] if hits else None
    tr_path, te_path = pick("train"), pick("test")
    if tr_path is None or te_path is None:
        raise FileNotFoundError(f"CICIoT2023 train/test CSV not found. CSVs found: {csvs[:10]}")
    print(f"  train file: {tr_path}\n  test file:  {te_path}")
    tr_raw, n_tr = sample_csv(tr_path, IOT_TRAIN_ROWS, SEED)
    te_raw, n_te = sample_csv(te_path, IOT_TEST_ROWS, SEED + 1000)
    train, test = prep_iot(tr_raw, tr_path), prep_iot(te_raw, te_path)
    train = train.drop_duplicates().reset_index(drop=True)
    stats = dict(raw_rows=n_tr + n_te, sampled_train=len(train), sampled_test=len(test),
                 split="official train.csv / test.csv (random subsample)", files=2)
    return train, test, [], stats

def read_kdd(path):
    df = pd.read_csv(path, header=None)
    if df.shape[1] == 43:
        df.columns = KDD_FEATURES + ["attack_type", "difficulty"]
        df = df.drop(columns=["difficulty"])
    elif df.shape[1] == 42:
        df.columns = KDD_FEATURES + ["attack_type"]
    else:
        raise ValueError(f"{path}: expected 42 or 43 columns, found {df.shape[1]}")
    df["attack_type"] = df["attack_type"].astype(str).str.strip().str.rstrip(".")
    df["y"] = (df["attack_type"] != "normal").astype(int)
    for c in KDD_CAT:
        df[c] = df[c].astype(str)
    for c in KDD_FEATURES:
        if c not in KDD_CAT:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(np.float32)
    return df.dropna().reset_index(drop=True)

def load_kdd(root):
    tr = find_files(root, "KDDTrain+.txt")
    te = find_files(root, "KDDTest+.txt")
    if not tr:
        raise FileNotFoundError(f"KDDTrain+.txt not found. .txt files: {find_files(root, '*.txt')}")
    print(f"  train file: {tr[0]}")
    train = read_kdd(tr[0])
    if te:
        print(f"  test file:  {te[0]}")
        test, split = read_kdd(te[0]), "official KDDTrain+ / KDDTest+"
    else:
        print("  WARNING: KDDTest+.txt missing -> stratified 70/30 split of KDDTrain+")
        train, test = train_test_split(train, test_size=0.3, stratify=train["y"], random_state=SEED)
        split = "stratified 70/30 of KDDTrain+ (KDDTest+ missing)"
    stats = dict(raw_rows=len(train) + len(test), split=split, files=2 if te else 1)
    return train, test, list(KDD_CAT), stats

LOADERS = {"CIC-IDS2017": load_cic, "CICIoT2023": load_iot, "NSL-KDD": load_kdd}

# ---------------------------------------------------------------------
# 4c. Run
# ---------------------------------------------------------------------
unzip_all()
DATA, t1_rows = {}, []
for name, loader in LOADERS.items():
    print(f"\nLoading {name} ...")
    try:
        train, test, cat_cols, stats = loader(os.path.join(DATA_ROOT, name))
    except Exception as e:
        raise RuntimeError(f"Loading {name} failed: {type(e).__name__}: {e}") from e
    feat_cols = [c for c in train.columns if c not in ("attack_type", "y")]
    feat_cols, dropped = drop_useless_columns(train, feat_cols, cat_cols)
    num_cols = [c for c in feat_cols if c not in cat_cols]
    for part_name, part in (("train", train), ("test", test)):
        if part["y"].nunique() < 2:
            raise ValueError(f"{name} {part_name} split has only one class: {part['y'].value_counts().to_dict()}")
    DATA[name] = dict(train=train.reset_index(drop=True), test=test.reset_index(drop=True),
                      num_cols=num_cols, cat_cols=cat_cols, feat_cols=num_cols + cat_cols,
                      dropped=dropped, stats=stats)
    t1_rows.append({
        "Dataset": name, "Split": stats["split"],
        "Train flows": len(train), "Test flows": len(test),
        "Features used": len(num_cols) + len(cat_cols), "Categorical features": len(cat_cols),
        "Columns dropped (constant/duplicate)": len(dropped),
        "Attack % (train)": round(100 * train["y"].mean(), 2),
        "Attack % (test)": round(100 * test["y"].mean(), 2),
        "Attack types (train)": train["attack_type"].nunique(),
    })
    print(f"  train={len(train):,}  test={len(test):,}  features={len(feat_cols)}  "
          f"dropped={len(dropped)}  attack% train/test = {100*train['y'].mean():.1f}/{100*test['y'].mean():.1f}")
    gc.collect()

table1 = pd.DataFrame(t1_rows)
save_table(table1, "T1_datasets")
with open(out("tables", "T1_preprocessing_details.json"), "w") as f:
    json.dump({n: dict(stats=d["stats"], dropped=d["dropped"]) for n, d in DATA.items()},
              f, indent=2, default=str)
print("\nAll datasets loaded:", list(DATA))
table1

In [ ]:
# =====================================================================
# 5. REFLEX KERNEL (Random Forest)  ->  Table 2, delta sweep, Fig. reflex routing
# =====================================================================
def make_X(ds, df):
    X = df[ds["num_cols"]].to_numpy(np.float32)
    if ds["cat_cols"]:
        Xc = ds["encoder"].transform(df[ds["cat_cols"]].astype(str)).astype(np.float32)
        X = np.hstack([X, Xc])
    return X

def cls_metrics(y, p):
    y, p = np.asarray(y).astype(int), np.asarray(p).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(acc=accuracy_score(y, p), f1=f1_score(y, p, zero_division=0),
                fnr=fn / (tp + fn) if (tp + fn) else np.nan,
                fpr=fp / (tn + fp) if (tn + fp) else np.nan)

REFLEX_LAT = {}   # median single-flow latency (s) per dataset
t2_rows, dsweep_rows = [], []
for name, ds in DATA.items():
    tr, te = ds["train"], ds["test"]
    if ds["cat_cols"]:
        ds["encoder"] = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        ds["encoder"].fit(tr[ds["cat_cols"]].astype(str))
    Xtr, Xte = make_X(ds, tr), make_X(ds, te)
    rf = RandomForestClassifier(**RF_PARAMS)
    t0 = time.perf_counter(); rf.fit(Xtr, tr["y"].values); fit_s = time.perf_counter() - t0
    assert list(rf.classes_) == [0, 1]
    p_oob = rf.oob_decision_function_[:, 1]
    tr["p_oob"] = p_oob
    tr["u_oob"] = np.abs(p_oob - 0.5)
    tr["y_reflex"] = (p_oob >= 0.5).astype(int)
    proba = rf.predict_proba(Xte)[:, 1]
    te["p_reflex"] = proba
    te["u"] = np.abs(proba - 0.5)
    te["y_reflex"] = (proba >= 0.5).astype(int)
    ds["rf"] = rf

    # single-flow latency, as a deployed reflex would see it
    rf.n_jobs = 1
    rng = np.random.default_rng(SEED)
    lat = []
    for i in rng.integers(0, len(Xte), 200):
        t0 = time.perf_counter(); rf.predict_proba(Xte[i:i + 1]); lat.append(time.perf_counter() - t0)
    rf.n_jobs = RF_PARAMS.get("n_jobs", None)
    REFLEX_LAT[name] = float(np.median(lat))

    imp = pd.Series(rf.feature_importances_, index=ds["feat_cols"]).sort_values(ascending=False)
    ds["prompt_feats"] = list(imp.index[:FEATURE_TOP_K]) if FEATURE_TOP_K else ds["feat_cols"]
    imp.reset_index().rename(columns={"index": "feature", 0: "importance"}).to_csv(
        out("tables", f"RF_feature_importance_{name}.csv"), index=False)

    m = cls_metrics(te["y"], te["y_reflex"])
    routed = te["u"] <= DELTA
    t2_rows.append({
        "Dataset": name, "Accuracy": m["acc"], "F1 (attack)": m["f1"], "FNR": m["fnr"], "FPR": m["fpr"],
        "Train time (s)": fit_s, "Median latency per flow (ms)": 1000 * REFLEX_LAT[name],
        f"Routed to LLM at delta={DELTA} (n)": int(routed.sum()),
        f"Routed to LLM at delta={DELTA} (%)": 100 * routed.mean(),
        "Reflex accuracy on accepted": accuracy_score(te.loc[~routed, "y"], te.loc[~routed, "y_reflex"]) if (~routed).any() else np.nan,
        "Reflex accuracy on routed": accuracy_score(te.loc[routed, "y"], te.loc[routed, "y_reflex"]) if routed.any() else np.nan,
    })
    for d in DELTA_SWEEP:
        r = te["u"] <= d
        dsweep_rows.append({"Dataset": name, "delta": d, "Routed (%)": 100 * r.mean(), "Routed (n)": int(r.sum()),
                            "Reflex acc on accepted": accuracy_score(te.loc[~r, "y"], te.loc[~r, "y_reflex"]) if (~r).any() else np.nan})
    print(f"{name}: acc={m['acc']:.4f} F1={m['f1']:.4f} FNR={m['fnr']:.4f} routed={routed.sum()} ({100*routed.mean():.3f}%)")

table2 = pd.DataFrame(t2_rows); save_table(table2, "T2_reflex_kernel")
table_delta = pd.DataFrame(dsweep_rows); save_table(table_delta, "T2b_delta_sweep")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
for name, g in table_delta.groupby("Dataset"):
    ax.plot(g["delta"], g["Routed (%)"].clip(lower=1e-3), marker="o", label=name)
ax.axvline(DELTA, ls="--", c="grey", lw=1)
ax.set_yscale("log"); ax.set_xlabel(r"Confidence threshold $\delta$")
ax.set_ylabel("Test flows routed to the LLM (%)"); ax.set_title("Reflex routing vs. $\\delta$")
ax.legend(); save_fig(fig, "F_reflex_routing_vs_delta"); plt.show()
table2

In [ ]:
# =====================================================================
# 6. LLM EVALUATION SETS + PROMPTS  (phi(x), Eq. 2)
#    - routed : test flows the reflex is unsure about (u <= DELTA)  -> realistic cascade
#    - stress : balanced random test flows (not routed)              -> enough samples to
#               measure the breaker
#    - calib  : TRAINING flows the cascade would route (out-of-bag u <= DELTA) -> used only
#               to choose tau, so no test flow ever influences tau
# =====================================================================
SYSTEM_PROMPT = ("You are a network intrusion detection analyst. You receive the feature values "
                 "of one network flow and decide whether it is benign traffic or an attack.")
ZS_TEMPLATE = ("Network flow features:\n{features}\n\n"
               "Is this flow BENIGN or ATTACK? First explain your reasoning in two to four short sentences, "
               "referring to the feature values that matter most. Only after that, write the last line "
               "exactly as 'Final answer: BENIGN' or 'Final answer: ATTACK'.")
RAG_TEMPLATE = ("Labelled reference flows from the training data (the most similar benign flows and the most "
                "similar attack flows, in random order):\n\n{references}\n\n"
                "Flow to classify:\n{features}\n\n"
                "Is the flow to classify BENIGN or ATTACK? First compare it with the reference flows and explain "
                "your reasoning in two to four short sentences, referring to the feature values that matter most. "
                "Only after that, write the last line exactly as 'Final answer: BENIGN' or 'Final answer: ATTACK'.")
USER_TEMPLATE = RAG_TEMPLATE if PROMPT_MODE == "retrieval" else ZS_TEMPLATE
# The assistant's reply is started with this text (already inside a sentence), so the model has to
# reason before it can reach the answer line.
ASSISTANT_PREFIX = ("Reasoning: Comparing the flow with the reference flows," if PROMPT_MODE == "retrieval"
                    else "Reasoning: Looking at the feature values,")

def fmt_val(v):
    if isinstance(v, str):
        return v
    v = float(v)
    if v.is_integer() and abs(v) < 1e15:
        return str(int(v))
    return f"{v:.4g}"

def feat_lines(row, feats):
    return "\n".join(f"- {f}: {fmt_val(row[f])}" for f in feats)

def signed_log(a):
    return np.sign(a) * np.log1p(np.abs(a))

class RefIndex:
    """Nearest labelled TRAINING flows per class, on the prompt features
    (signed-log + standardised numeric features, one-hot categorical features)."""
    def __init__(self, ds, feats):
        from sklearn.neighbors import NearestNeighbors
        tr = ds["train"]
        self.train, self.feats = tr, feats
        self.num = [f for f in feats if f not in ds["cat_cols"]]
        self.cat = [f for f in feats if f in ds["cat_cols"]]
        Xn = signed_log(tr[self.num].to_numpy(np.float64))
        self.mu, self.sd = Xn.mean(0), Xn.std(0) + 1e-9
        if self.cat:
            self.dummy_cols = pd.get_dummies(tr[self.cat].astype(str)).columns
        r = np.random.default_rng(SEED)
        self.pools = {}
        for cls in (0, 1):
            idx = np.flatnonzero(tr["y"].values == cls)
            if len(idx) > REF_POOL_PER_CLASS:
                idx = np.sort(r.choice(idx, REF_POOL_PER_CLASS, replace=False))
            nn = NearestNeighbors(n_neighbors=min(N_REF_PER_CLASS + 1, len(idx))).fit(self.embed(tr.loc[idx]))
            self.pools[cls] = (idx, nn)

    def embed(self, df):
        parts = [(signed_log(df[self.num].to_numpy(np.float64)) - self.mu) / self.sd]
        if self.cat:
            parts.append(pd.get_dummies(df[self.cat].astype(str)).reindex(
                columns=self.dummy_cols, fill_value=0).to_numpy(np.float64))
        return np.hstack(parts)

    def query(self, df, exclude=None):
        """Returns, per row, a list of (train_index, label). `exclude`: the row's own train index (calibration flows)."""
        E = self.embed(df)
        res = {cls: self.pools[cls][1].kneighbors(E, return_distance=False) for cls in (0, 1)}
        out = []
        for i in range(len(df)):
            refs = []
            for cls in (0, 1):
                idx = self.pools[cls][0]
                cand = [int(idx[k]) for k in res[cls][i] if exclude is None or idx[k] != exclude[i]]
                refs += [(c, cls) for c in cand[:N_REF_PER_CLASS]]
            out.append(refs)
        return out

def build_prompt(row, feats, refs=None, train=None, seed=0):
    if PROMPT_MODE != "retrieval" or not refs:
        return ZS_TEMPLATE.format(features=feat_lines(row, feats))
    order = np.random.default_rng(seed).permutation(len(refs))
    blocks = [f"Reference flow {n + 1} ({'ATTACK' if refs[o][1] else 'BENIGN'}):\n"
              f"{feat_lines(train.loc[refs[o][0]], feats)}" for n, o in enumerate(order)]
    return RAG_TEMPLATE.format(references="\n\n".join(blocks), features=feat_lines(row, feats))

import hashlib
RUN_VERSION = hashlib.md5(json.dumps([SYSTEM_PROMPT, USER_TEMPLATE, ASSISTANT_PREFIX, MAX_NEW_TOKENS,
                                      FEATURE_TOP_K, DELTA, SEED, PROMPT_MODE, N_REF_PER_CLASS,
                                      REF_POOL_PER_CLASS]).encode()).hexdigest()[:10]
print("Run version (prompt + generation settings fingerprint):", RUN_VERSION)

rng = np.random.default_rng(SEED)
ev_parts = []
for name, ds in DATA.items():
    te = ds["test"]
    routed_idx = np.flatnonzero(te["u"].values <= DELTA)
    if len(routed_idx) > ROUTED_CAP:
        routed_idx = np.sort(rng.choice(routed_idx, ROUTED_CAP, replace=False))
    pool = te.drop(index=np.flatnonzero(te["u"].values <= DELTA))
    stress_idx = []
    for cls in (0, 1):
        cand = pool.index[pool["y"] == cls].values
        stress_idx += list(rng.choice(cand, min(len(cand), N_STRESS // 2), replace=False))
    stress_idx = np.array(sorted(stress_idx))

    tr = ds["train"]
    ok = np.isfinite(tr["u_oob"].values)
    cal_pool = np.flatnonzero(ok & (tr["u_oob"].values <= DELTA))
    n_unc = len(cal_pool)
    if len(cal_pool) > N_CALIB:
        cal_idx = rng.choice(cal_pool, N_CALIB, replace=False)
    else:   # too few uncertain training flows: top up with the least confident remaining ones
        rest = np.setdiff1d(np.flatnonzero(ok), cal_pool)
        rest = rest[np.argsort(tr["u_oob"].values[rest])][: N_CALIB - len(cal_pool)]
        cal_idx = np.concatenate([cal_pool, rest])
    cal_idx = np.sort(cal_idx)
    ref_index = RefIndex(ds, ds["prompt_feats"]) if PROMPT_MODE == "retrieval" else None

    for subset, src, idx, prefix in [("routed", te, routed_idx, ""), ("stress", te, stress_idx, ""),
                                     ("calib", tr, cal_idx, "train")]:
        part = src.loc[idx].copy()
        if subset == "calib":
            part["p_reflex"], part["u"] = part["p_oob"], part["u_oob"]
        part["subset"] = subset
        part["split"] = "calib" if subset == "calib" else "test"
        part["dataset"] = name
        part["sample_id"] = [f"{name}#{prefix}{i}" for i in part.index]
        if ref_index is not None:
            refs = ref_index.query(part, exclude=part.index.values if subset == "calib" else None)
        else:
            refs = [None] * len(part)
        part["prompt"] = [build_prompt(r, ds["prompt_feats"], rf_, tr, seed=SEED + int(i))
                          for (i, r), rf_ in zip(part.iterrows(), refs)]
        ev_parts.append(part[["sample_id", "dataset", "subset", "split", "y", "attack_type",
                              "p_reflex", "u", "y_reflex", "prompt"]])
    print(f"{name}: routed={len(routed_idx)} (of {int((te['u'] <= DELTA).sum())} uncertain test flows)  "
          f"stress={len(stress_idx)}  calib={len(cal_idx)} training flows "
          f"({n_unc} with OOB u<=delta{'' if n_unc >= N_CALIB else ', topped up with least-confident flows'})")

EVAL = pd.concat(ev_parts, ignore_index=True)
EVAL.to_csv(out("data", "eval_set.csv"), index=False)
print("\nEvaluation flows:", len(EVAL), "| saved to data/eval_set.csv")
print("\nExample prompt:\n" + EVAL["prompt"].iloc[0])
EVAL.groupby(["dataset", "subset", "split"]).size().rename("n").reset_index()

In [ ]:
# =====================================================================
# 7. LLM ENGINE: greedy decoding with per-token entropy and the circuit breaker
#
#    Written as an explicit token-by-token loop (with KV cache) so every step is visible:
#      for each new token t:
#        H_t   = entropy of the full next-token distribution (nats, from raw logits)   (Eq. 9)
#        token = argmax                                                           (greedy)
#        stop if EOS                                        -> reason "eos"
#        stop if 'Final answer: X' has been produced        -> reason "completed"
#        stop if t >= MIN_TOKENS and mean(H_{t-k+1..t}) > tau -> reason "tripped" (Eq. 10-12)
#        stop if t == MAX_NEW_TOKENS                        -> reason "budget"    (Eq. 13)
#    With tau=None the breaker is off: that is the "naive" system.
#    Because decoding is greedy, an EGCB run is exactly a prefix of the naive run.
# =====================================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None
torch.manual_seed(SEED)

ANSWER_RE = re.compile(r"final\s+answer\s*[:\-]?\s*\**\s*(benign|attack|malicious|normal)", re.I)
ANSWER_MAP = {"benign": "benign", "normal": "benign", "attack": "attack", "malicious": "attack"}

def env_info():
    info = dict(python=platform.python_version(), torch=torch.__version__,
                transformers=__import__("transformers").__version__, sklearn=sklearn.__version__,
                pandas=pd.__version__, cpu=platform.processor() or platform.machine(),
                cpu_count=os.cpu_count())
    try:
        import psutil; info["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except Exception:
        pass
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update(gpu=p.name, gpu_mem_gb=round(p.total_memory / 1e9, 1), cuda=torch.version.cuda)
    else:
        info["gpu"] = "none (CPU)"
    return info

ENV = env_info()
with open(out("environment.json"), "w") as f:
    json.dump(ENV, f, indent=2)
print(json.dumps(ENV, indent=2))

def pick_dtype():
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32

def load_llm(mcfg):
    hf_id = mcfg["hf_id"]
    tok = AutoTokenizer.from_pretrained(hf_id)
    dtype = pick_dtype()
    kwargs = dict(torch_dtype=dtype)
    use_4bit = False
    if torch.cuda.is_available():
        gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        need_gb = mcfg["params_b"] * 2.1 + 2.0
        use_4bit = FORCE_4BIT or need_gb > gpu_gb
        kwargs["device_map"] = {"": 0}          # whole model on the GPU; never silent CPU offload
        if use_4bit:
            if BitsAndBytesConfig is None:
                raise RuntimeError("bitsandbytes not available for 4-bit loading")
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16 if dtype == torch.float32 else dtype)
    model = AutoModelForCausalLM.from_pretrained(hf_id, **kwargs).eval()
    eos = set()
    for e in (tok.eos_token_id, getattr(model.generation_config, "eos_token_id", None)):
        if e is not None:
            eos.update(e if isinstance(e, (list, tuple)) else [e])
    precision = "4-bit NF4" if use_4bit else str(dtype).replace("torch.", "")
    print(f"Loaded {hf_id} | precision={precision} | device={next(model.parameters()).device}")
    return dict(tok=tok, model=model, eos=eos, precision=precision, hf_id=hf_id)

def unload_llm(bundle):
    del bundle["model"]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def encode_prompt(tok, user_text):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_text}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + ASSISTANT_PREFIX
    return tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"]

@torch.no_grad()
def generate_with_breaker(bundle, user_text, tau=None, window=WINDOW,
                          min_tokens=MIN_TOKENS, max_new_tokens=MAX_NEW_TOKENS):
    tok, model = bundle["tok"], bundle["model"]
    dev = next(model.parameters()).device
    ids = encode_prompt(tok, user_text).to(dev)
    gen, H, T = [], [], []
    reason, answer, answer_step = "budget", None, None

    t0 = time.perf_counter()
    out_ = model(input_ids=ids, use_cache=True)
    past, logits = out_.past_key_values, out_.logits[:, -1, :]
    for t in range(1, max_new_tokens + 1):
        lf = logits.float()
        logp = torch.log_softmax(lf, dim=-1)
        h = float(-(logp.exp() * logp).sum(dim=-1).item())
        nxt = int(torch.argmax(lf, dim=-1).item())       # .item() synchronises the GPU
        T.append(time.perf_counter() - t0)
        gen.append(nxt); H.append(h)
        if nxt in bundle["eos"]:
            reason = "eos"; break
        m = ANSWER_RE.search(tok.decode(gen, skip_special_tokens=True))
        if m:
            reason, answer, answer_step = "completed", ANSWER_MAP[m.group(1).lower()], t; break
        if tau is not None and t >= min_tokens and np.mean(H[-window:]) > tau:
            reason = "tripped"; break
        if t == max_new_tokens:
            break
        out_ = model(input_ids=torch.tensor([[nxt]], device=dev), past_key_values=past, use_cache=True)
        past, logits = out_.past_key_values, out_.logits[:, -1, :]

    return dict(prompt_tokens=int(ids.shape[1]), gen_tokens=len(gen),
                entropies=[round(x, 5) for x in H], t_cum=[round(x, 5) for x in T],
                total_s=T[-1] if T else 0.0, stop_reason=reason, llm_answer=answer,
                answer_step=answer_step, nonfinite_entropy=bool(not np.all(np.isfinite(H))),
                text=tok.decode(gen, skip_special_tokens=True))

def warmup(bundle):
    for _ in range(2):
        generate_with_breaker(bundle, "Reply with OK.", tau=None, max_new_tokens=8)
    # show one real generation so a broken prompt is noticed immediately
    demo = generate_with_breaker(bundle, EVAL["prompt"].iloc[0], tau=None)
    print(f"  demo: {demo['gen_tokens']} tokens, stop={demo['stop_reason']}, answer={demo['llm_answer']}")
    print("  " + (ASSISTANT_PREFIX + demo["text"])[:500].replace("\n", "\n  "))

In [ ]:
# =====================================================================
# 8. RUN HELPERS (resumable: every finished sample is appended to a JSONL file)
# =====================================================================
def run_path(model_key, mode):
    return out("llm_runs", f"{model_key}__{mode}.jsonl")

def load_runs(model_key, mode, tau="any"):
    """Only returns records made with the current prompt/settings (RUN_VERSION) and, for EGCB, the same tau."""
    p = run_path(model_key, mode)
    recs, stale = {}, 0
    if os.path.exists(p):
        with open(p) as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                same_tau = tau == "any" or r.get("tau") == tau or (
                    r.get("tau") is not None and tau is not None and np.isclose(r["tau"], tau))
                if r.get("run_version") == RUN_VERSION and same_tau:
                    recs[r["sample_id"]] = r
                else:
                    stale += 1
    if stale:
        print(f"  [{model_key}/{mode}] ignoring {stale} old record(s) from a different prompt/settings/tau")
    return recs

def run_model_on(model_key, mcfg, rows, mode, tau_fn):
    """mode: 'naive' (breaker off) or 'nsacb' (breaker on with tau_fn(model_key))."""
    tau = tau_fn(model_key) if mode == "nsacb" else None
    done = load_runs(model_key, mode, tau=tau if mode == "nsacb" else "any")
    todo = rows[~rows["sample_id"].isin(done)]
    if todo.empty:
        print(f"[{model_key}/{mode}] all {len(rows)} samples already done"); return
    print(f"[{model_key}/{mode}] {len(todo)} to run ({len(done)} already done)")
    bundle = load_llm(mcfg); warmup(bundle)
    t_start = time.time()
    with open(run_path(model_key, mode), "a") as f:
        for j, r in enumerate(todo.itertuples(index=False), 1):
            rec = generate_with_breaker(bundle, r.prompt, tau=tau)
            rec.update(sample_id=r.sample_id, dataset=r.dataset, model=model_key, hf_id=mcfg["hf_id"],
                       run_version=RUN_VERSION,
                       precision=bundle["precision"], params_b=mcfg["params_b"], mode=mode,
                       tau=tau, window=WINDOW, min_tokens=MIN_TOKENS, max_new_tokens=MAX_NEW_TOKENS)
            f.write(json.dumps(rec) + "\n"); f.flush()
            if j % 10 == 0 or j == len(todo):
                el = time.time() - t_start
                print(f"  {j}/{len(todo)}  elapsed {el/60:.1f} min  ETA {el/j*(len(todo)-j)/60:.1f} min")
    unload_llm(bundle)

In [ ]:
# =====================================================================
# 9. NAIVE RUN: every model on every evaluation flow, breaker OFF
# =====================================================================
for mk, mcfg in MODELS.items():
    run_model_on(mk, mcfg, EVAL, "naive", tau_fn=None)

NAIVE = {mk: load_runs(mk, "naive") for mk in MODELS}
ytrue = dict(zip(EVAL["sample_id"], EVAL["y"]))
for mk, recs in NAIVE.items():
    rs = list(recs.values())
    reasons = pd.Series([r["stop_reason"] for r in rs]).value_counts().to_dict()
    answers = pd.Series([r["llm_answer"] for r in rs]).value_counts(dropna=False).to_dict()
    steps = [r["answer_step"] for r in rs if r["answer_step"]]
    acc = np.mean([(r["llm_answer"] == "attack") == bool(ytrue[r["sample_id"]]) for r in rs if r["llm_answer"]]) if steps else float("nan")
    print(f"{mk}: stop reasons {reasons} | answers {answers} | median answer at token "
          f"{np.median(steps) if steps else 'n/a'} | LLM accuracy when answered {acc:.3f} | "
          f"non-finite entropy runs {sum(r['nonfinite_entropy'] for r in rs)}")
    if steps and np.median(steps) <= MIN_TOKENS:
        print(f"  WARNING: {mk} answers before the breaker can act (median {np.median(steps)} <= MIN_TOKENS={MIN_TOKENS}).")

In [ ]:
# =====================================================================
# 10. OFFLINE REPLAY + TAU CALIBRATION (calibration split only)
#     Replaying the breaker on a naive trajectory gives exactly what a live EGCB run
#     would do (greedy decoding); the live run in step 11 checks this.
# =====================================================================
def smoothed_eligible(rec, window, min_tokens):
    H = np.asarray(rec["entropies"], dtype=float)
    n = len(H)
    c = np.concatenate([[0.0], np.cumsum(H)])
    t = np.arange(1, n + 1)
    lo = np.maximum(0, t - window)
    hbar = (c[t] - c[lo]) / (t - lo)
    elig = t >= min_tokens
    if rec["answer_step"]:
        elig &= t != rec["answer_step"]          # answer is checked before the breaker
    if rec["stop_reason"] == "eos":
        elig &= t != n                           # EOS is checked before the breaker
    return np.where(elig, hbar, -np.inf)

def replay(rec, tau, window=WINDOW, min_tokens=MIN_TOKENS, hb=None):
    trip = None
    if tau is not None and np.isfinite(tau):
        hb = smoothed_eligible(rec, window, min_tokens) if hb is None else hb
        idx = np.flatnonzero(hb > tau)
        trip = int(idx[0]) + 1 if idx.size else None
    if trip is None:
        return dict(reason=rec["stop_reason"], answer=rec["llm_answer"],
                    tokens=rec["gen_tokens"], latency=rec["total_s"])
    return dict(reason="tripped", answer=None, tokens=trip, latency=rec["t_cum"][trip - 1])

def to_pred(answer, y_reflex):
    """LLM answer if one was produced, otherwise FALLBACK to the reflex prediction."""
    if answer is None:
        return int(y_reflex)
    return 1 if answer == "attack" else 0

def outcome_rows(recs, rows, tau, window=WINDOW, min_tokens=MIN_TOKENS, live=False):
    res = []
    for r in rows.itertuples(index=False):
        rec = recs[r.sample_id]
        o = (dict(reason=rec["stop_reason"], answer=rec["llm_answer"], tokens=rec["gen_tokens"],
                  latency=rec["total_s"]) if live else replay(rec, tau, window, min_tokens))
        res.append(dict(sample_id=r.sample_id, dataset=r.dataset, subset=r.subset, y=int(r.y),
                        y_reflex=int(r.y_reflex), pred=to_pred(o["answer"], r.y_reflex),
                        llm_answered=o["answer"] is not None, reason=o["reason"],
                        tokens=o["tokens"], latency=o["latency"] + REFLEX_LAT[r.dataset]))
    return pd.DataFrame(res)

def select_tau(recs, rows, window, min_tokens):
    hbs = {sid: smoothed_eligible(recs[sid], window, min_tokens) for sid in rows["sample_id"]}
    vals = np.concatenate([h[np.isfinite(h)] for h in hbs.values()]) if hbs else np.array([])
    grid = np.unique(np.quantile(vals, np.linspace(0, 1, 101))) if vals.size else np.array([])
    grid = np.append(grid, np.inf)
    ys, yref = rows["y"].values, rows["y_reflex"].values
    table = []
    for tau in grid:
        preds, toks, trips = [], [], 0
        for sid, yr in zip(rows["sample_id"], yref):
            o = replay(recs[sid], tau, window, min_tokens, hb=hbs[sid])
            preds.append(to_pred(o["answer"], yr)); toks.append(o["tokens"]); trips += o["reason"] == "tripped"
        err = 1 - accuracy_score(ys, preds)
        cost = np.mean(toks) / MAX_NEW_TOKENS
        table.append(dict(tau=tau, error=err, norm_tokens=cost, trip_rate=trips / len(rows),
                          J=LAMBDA * err + (1 - LAMBDA) * cost))
    table = pd.DataFrame(table)
    best = table.loc[table["J"] == table["J"].min(), "tau"].max()   # ties -> least intervention
    return float(best), table

TAU_STAR, TAU_STAR_K1, TAU_TABLES = {}, {}, []
for mk in MODELS:
    cal = EVAL[EVAL["split"] == "calib"]
    if TAU_FIXED is not None:
        TAU_STAR[mk] = TAU_STAR_K1[mk] = float(TAU_FIXED); continue
    TAU_STAR[mk], tab = select_tau(NAIVE[mk], cal, WINDOW, MIN_TOKENS)
    TAU_STAR_K1[mk], tab1 = select_tau(NAIVE[mk], cal, 1, MIN_TOKENS)
    tab["model"], tab["window"] = mk, WINDOW; tab1["model"], tab1["window"] = mk, 1
    TAU_TABLES += [tab, tab1]
    print(f"{mk}: tau* = {TAU_STAR[mk]:.4f} (k={WINDOW})   tau*_k1 = {TAU_STAR_K1[mk]:.4f} (k=1)")
if TAU_TABLES:
    save_table(pd.concat(TAU_TABLES, ignore_index=True), "T4_tau_sweep_calibration")
with open(out("tau_star.json"), "w") as f:
    json.dump({"tau_star": TAU_STAR, "tau_star_k1": TAU_STAR_K1}, f, indent=2)

In [ ]:
# =====================================================================
# 11. LIVE EGCB RUN on the test split (breaker really stops generation)
# =====================================================================
TEST = EVAL[EVAL["split"] == "test"].reset_index(drop=True)
LIVE = {}
if RUN_LIVE:
    for mk, mcfg in MODELS.items():
        run_model_on(mk, mcfg, TEST, "nsacb", tau_fn=lambda m: TAU_STAR[m])
        LIVE[mk] = load_runs(mk, "nsacb", tau=TAU_STAR[mk])

# Agreement between live EGCB and offline replay  ->  Table 8
t8 = []
for mk, recs in LIVE.items():
    same_reason = same_tokens = same_answer = 0
    for sid, lrec in recs.items():
        o = replay(NAIVE[mk][sid], TAU_STAR[mk])
        same_reason += o["reason"] == lrec["stop_reason"]
        same_tokens += o["tokens"] == lrec["gen_tokens"]
        same_answer += o["answer"] == lrec["llm_answer"]
    n = max(len(recs), 1)
    t8.append({"Model": mk, "n": len(recs), "Same stop reason (%)": 100 * same_reason / n,
               "Same token count (%)": 100 * same_tokens / n, "Same answer (%)": 100 * same_answer / n})
if t8:
    save_table(pd.DataFrame(t8), "T8_live_vs_replay_agreement")
    display(pd.DataFrame(t8))

In [ ]:
# =====================================================================
# 12. RESULTS TABLES
# =====================================================================
def boot_mean_ci(v, B=BOOTSTRAP, seed=SEED):
    v = np.asarray(v, dtype=float)
    if len(v) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    bs = v[rng.integers(0, len(v), (B, len(v)))].mean(axis=1)
    return (v.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5))

def ci_str(v, scale=1.0, nd=3):
    m, lo, hi = boot_mean_ci(v)
    return f"{m*scale:.{nd}f} [{lo*scale:.{nd}f}, {hi*scale:.{nd}f}]"

def summarize(o, label):
    m = cls_metrics(o["y"], o["pred"])
    return {"Method": label, "n": len(o),
            "Accuracy [95% CI]": ci_str((o["y"] == o["pred"]).astype(float)),
            "F1 (attack)": round(m["f1"], 3), "FNR": round(m["fnr"], 3),
            "LLM answered (%)": round(100 * o["llm_answered"].mean(), 1),
            "LLM accuracy when it answered": (round(float((o.loc[o["llm_answered"], "y"] == o.loc[o["llm_answered"], "pred"]).mean()), 3)
                                              if o["llm_answered"].any() else np.nan),
            "Entropy trips (%)": round(100 * (o["reason"] == "tripped").mean(), 1),
            "Token budget hit (%)": round(100 * (o["reason"] == "budget").mean(), 1),
            "Generated tokens [95% CI]": ci_str(o["tokens"], nd=1),
            "Latency s [95% CI]": ci_str(o["latency"], nd=2),
            "_acc": m["acc"], "_tokens": o["tokens"].mean(), "_lat": o["latency"].mean()}

OUT_NAIVE, OUT_NSACB = {}, {}
t3 = []
for mk in MODELS:
    OUT_NAIVE[mk] = outcome_rows(NAIVE[mk], TEST, None)
    OUT_NSACB[mk] = (outcome_rows(LIVE[mk], TEST, None, live=True) if mk in LIVE
                     else outcome_rows(NAIVE[mk], TEST, TAU_STAR[mk]))
    for ds_name in list(DATA) + ["ALL"]:
        for subset in ["routed", "stress", "all"]:
            sel = lambda o: o[((o["dataset"] == ds_name) | (ds_name == "ALL")) &
                              ((o["subset"] == subset) | (subset == "all"))]
            a, b = sel(OUT_NAIVE[mk]), sel(OUT_NSACB[mk])
            if len(a) == 0:
                continue
            sa, sb = summarize(a, "Naive"), summarize(b, "EGCB")
            for s in (sa, sb):
                s.update(Model=mk, Dataset=ds_name, Subset=subset)
            sa["tau"], sb["tau"] = "off", round(TAU_STAR[mk], 4)
            sb["Token saving (%)"] = round(100 * (1 - sb["_tokens"] / sa["_tokens"]), 1)
            sb["Latency saving (%)"] = round(100 * (1 - sb["_lat"] / sa["_lat"]), 1)
            t3 += [sa, sb]
table3 = pd.DataFrame(t3)
cols = ["Model", "Dataset", "Subset", "Method", "tau", "n", "Accuracy [95% CI]", "F1 (attack)", "FNR",
        "LLM answered (%)", "LLM accuracy when it answered", "Entropy trips (%)", "Token budget hit (%)", "Generated tokens [95% CI]",
        "Latency s [95% CI]", "Token saving (%)", "Latency saving (%)"]
table3 = table3.reindex(columns=cols)
save_table(table3, "T3_main_results_naive_vs_nsacb")

# ---------------- Table 5: ablation and baselines (test split) ----------------
def fixed_budget(recs, rows, B):
    res = []
    for r in rows.itertuples(index=False):
        rec = recs[r.sample_id]; n = rec["gen_tokens"]
        if rec["answer_step"] and rec["answer_step"] <= B:
            ans, tok = rec["llm_answer"], rec["answer_step"]
        else:
            ans, tok = None, min(n, B)
        res.append(dict(dataset=r.dataset, subset=r.subset, y=int(r.y), pred=to_pred(ans, r.y_reflex),
                        llm_answered=ans is not None,
                        reason="budget" if ans is None and n > B else ("completed" if ans else rec["stop_reason"]),
                        tokens=tok, latency=rec["t_cum"][tok - 1] + REFLEX_LAT[r.dataset]))
    return pd.DataFrame(res)

def random_breaker(recs, rows, ns_out, repeats=RANDOM_TRIP_REPEATS, seed=SEED):
    """Trips the same number of samples as EGCB, chosen at random, at EGCB's trip steps."""
    trip_steps = ns_out.loc[ns_out["reason"] == "tripped", "tokens"].values
    k = len(trip_steps)
    rng = np.random.default_rng(seed)
    accs, toks, lats = [], [], []
    sids = rows["sample_id"].values
    for _ in range(repeats):
        chosen = set(rng.choice(len(rows), k, replace=False)) if k else set()
        correct, tk, lt = [], [], []
        for i, r in enumerate(rows.itertuples(index=False)):
            rec = recs[sids[i]]; n = rec["gen_tokens"]
            ans, tok = rec["llm_answer"], n
            if i in chosen:
                s = int(min(n, rng.choice(trip_steps)))
                if not (rec["answer_step"] and rec["answer_step"] <= s):
                    ans, tok = None, s
            correct.append(to_pred(ans, r.y_reflex) == r.y); tk.append(tok)
            lt.append(rec["t_cum"][tok - 1] + REFLEX_LAT[r.dataset])
        accs.append(np.mean(correct)); toks.append(np.mean(tk)); lats.append(np.mean(lt))
    return np.mean(accs), np.percentile(accs, [2.5, 97.5]), np.mean(toks), np.mean(lats)

t5 = []
for mk in MODELS:
    for ds_name in list(DATA) + ["ALL"]:
        rows = TEST[(TEST["dataset"] == ds_name) | (ds_name == "ALL")]
        if rows.empty:
            continue
        pick = lambda o: o[o["sample_id"].isin(rows["sample_id"])] if "sample_id" in o else o
        nv, ns = pick(OUT_NAIVE[mk]), pick(OUT_NSACB[mk])
        k1 = outcome_rows(NAIVE[mk], rows, TAU_STAR_K1[mk], window=1)
        B = max(1, int(round(ns["tokens"].mean())))
        fb = fixed_budget(NAIVE[mk], rows, B)
        reflex = nv.assign(pred=nv["y_reflex"], llm_answered=False, reason="reflex", tokens=0,
                           latency=[REFLEX_LAT[d] for d in nv["dataset"]])
        entries = [("Reflex only (RF)", reflex), ("LLM naive (no breaker)", nv),
                   (f"EGCB (k={WINDOW})", ns), ("EGCB without smoothing (k=1)", k1),
                   (f"Fixed token budget (B={B}, same mean compute)", fb)]
        for label, o in entries:
            s = summarize(o, label); s.update(Model=mk, Dataset=ds_name); t5.append(s)
        ra, rci, rt, rl = random_breaker(NAIVE[mk], rows, ns)
        t5.append({"Model": mk, "Dataset": ds_name, "Method": "Random breaker (same trip rate)",
                   "n": len(rows), "Accuracy [95% CI]": f"{ra:.3f} [{rci[0]:.3f}, {rci[1]:.3f}]",
                   "Generated tokens [95% CI]": f"{rt:.1f}", "Latency s [95% CI]": f"{rl:.2f}",
                   "_acc": ra, "_tokens": rt, "_lat": rl})
table5 = pd.DataFrame(t5)
save_table(table5.reindex(columns=["Model", "Dataset", "Method", "n", "Accuracy [95% CI]", "F1 (attack)",
                                   "FNR", "LLM answered (%)", "LLM accuracy when it answered", "Entropy trips (%)",
                                   "Generated tokens [95% CI]", "Latency s [95% CI]"]), "T5_ablation_baselines")

# ---------------- Table 6: does entropy predict LLM failures? ----------------
# Three failure definitions are reported separately, because they answer different questions:
#   any_failure : no answer OR wrong answer                (all flows)
#   wrong_answer: wrong answer, among flows that answered  (quality of the reasoning itself)
#   no_answer   : ran out of tokens / stopped without answer (all flows)
# AUROC > 0.5: higher entropy goes with failure (what the breaker assumes).
# AUROC < 0.5: failures are the LOW-entropy (confident) outputs - the breaker would cut the wrong ones.
def auroc_ci(y, s, B=BOOTSTRAP, seed=SEED):
    y, s = np.asarray(y), np.asarray(s)
    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed); bs = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            bs.append(roc_auc_score(y[i], s[i]))
    lo, hi = (np.percentile(bs, 2.5), np.percentile(bs, 97.5)) if bs else (np.nan, np.nan)
    return roc_auc_score(y, s), lo, hi

def fmt_auc(a, lo, hi):
    return "n/a (only one outcome)" if np.isnan(a) else f"{a:.3f} [{lo:.3f}, {hi:.3f}]"

EARLY_TOKENS = 32
t6 = []
for mk in MODELS:
    for ds_name in list(DATA) + ["ALL"]:
        rows = TEST[(TEST["dataset"] == ds_name) | (ds_name == "ALL")]
        recs = [(NAIVE[mk][r.sample_id], int(r.y)) for r in rows.itertuples(index=False)]
        answered = [rec["llm_answer"] is not None for rec, _ in recs]
        wrong = [rec["llm_answer"] is not None and (rec["llm_answer"] == "attack") != bool(y) for rec, y in recs]
        early = np.array([np.mean(rec["entropies"][:EARLY_TOKENS]) for rec, _ in recs])
        full = np.array([np.mean(rec["entropies"]) for rec, _ in recs])
        answered, wrong = np.array(answered), np.array(wrong)
        defs = {
            "any_failure (no answer or wrong)": (np.ones(len(recs), bool), (~answered) | wrong),
            "wrong_answer (answered flows only)": (answered, wrong),
            "no_answer (all flows)": (np.ones(len(recs), bool), ~answered),
        }
        for dname, (mask, fail) in defs.items():
            y_, e_, f_ = fail[mask].astype(int), early[mask], full[mask]
            a1 = auroc_ci(y_, e_); a2 = auroc_ci(y_, f_)
            t6.append({"Model": mk, "Dataset": ds_name, "Failure definition": dname,
                       "n": int(mask.sum()), "Failures (n)": int(y_.sum()),
                       f"AUROC mean entropy, first {EARLY_TOKENS} tokens [95% CI]": fmt_auc(*a1),
                       "AUROC mean entropy, full output [95% CI]": fmt_auc(*a2),
                       "Mean early entropy: failures": round(float(e_[y_ == 1].mean()), 3) if y_.sum() else np.nan,
                       "Mean early entropy: successes": round(float(e_[y_ == 0].mean()), 3) if (y_ == 0).sum() else np.nan,
                       "_auc_early": a1[0], "_auc_full": a2[0]})
table6 = pd.DataFrame(t6)
save_table(table6.drop(columns=["_auc_early", "_auc_full"]), "T6_entropy_predicts_failures_auroc")

# ---------------- Table 7: end-to-end cascade on the full test split ----------------
t7 = []
for mk in MODELS:
    for name, ds in DATA.items():
        te = ds["test"]; routed = (te["u"] <= DELTA).values
        n_all, n_r = len(te), int(routed.sum())
        o_n = OUT_NAIVE[mk][(OUT_NAIVE[mk]["dataset"] == name) & (OUT_NAIVE[mk]["subset"] == "routed")]
        o_s = OUT_NSACB[mk][(OUT_NSACB[mk]["dataset"] == name) & (OUT_NSACB[mk]["subset"] == "routed")]
        acc_accept = accuracy_score(te.loc[~routed, "y"], te.loc[~routed, "y_reflex"]) if (~routed).any() else np.nan
        acc_reflex = accuracy_score(te["y"], te["y_reflex"])
        row = {"Model": mk, "Dataset": name, "Test flows": n_all, "Routed to LLM (n)": n_r,
               "Routed (%)": round(100 * n_r / n_all, 3), "Routed evaluated (n)": len(o_s),
               "Exact (all routed evaluated)": len(o_s) == n_r, "Reflex-only accuracy": round(acc_reflex, 5)}
        for lab, o in (("Cascade + naive LLM", o_n), ("Cascade + EGCB", o_s)):
            if n_r == 0 or len(o) == 0:
                row[f"{lab} accuracy"] = round(acc_reflex, 5); row[f"{lab} LLM tokens per 1k flows"] = 0.0
                continue
            acc_r = (o["y"] == o["pred"]).mean()
            est = (acc_accept * (n_all - n_r) + acc_r * n_r) / n_all if n_all > n_r else acc_r
            row[f"{lab} accuracy"] = round(est, 5)
            row[f"{lab} LLM tokens per 1k flows"] = round(1000 * n_r / n_all * o["tokens"].mean(), 2)
        t7.append(row)
table7 = pd.DataFrame(t7)
save_table(table7, "T7_cascade_end_to_end")

display(table3[(table3["Subset"] == "all")])
display(table5.drop(columns=["_acc", "_tokens", "_lat"]))
display(table6[table6["Dataset"] == "ALL"].drop(columns=["_auc_early", "_auc_full"]))
display(table7)

In [ ]:
# =====================================================================
# 13. FIGURES (all from measured data; every axis labelled)
# =====================================================================
MK = list(MODELS); DS = list(DATA)
def model_label(mk):
    return f"{mk} ({MODELS[mk]['params_b']:.1f}B)"

# (a) Entropy trajectories on test flows (full naive run; x = where EGCB trips)
fig, axes = plt.subplots(1, len(MK), figsize=(4.2 * len(MK), 3.4), squeeze=False)
for ax, mk in zip(axes[0], MK):
    shown = {"tripped": 0, "completed": 0}
    for r in TEST.itertuples(index=False):
        rec = NAIVE[mk][r.sample_id]; o = replay(rec, TAU_STAR[mk])
        key = "tripped" if o["reason"] == "tripped" else ("completed" if o["reason"] == "completed" else None)
        if key is None or shown[key] >= 5:
            continue
        H = np.asarray(rec["entropies"]); c = np.cumsum(np.insert(H, 0, 0)); t = np.arange(1, len(H) + 1)
        lo = np.maximum(0, t - WINDOW); hbar = (c[t] - c[lo]) / (t - lo)
        col = "tab:red" if key == "tripped" else "tab:blue"
        ax.plot(t, hbar, color=col, alpha=0.7, lw=1.2,
                label=("EGCB trips" if key == "tripped" else "EGCB completes") if shown[key] == 0 else None)
        if key == "tripped":
            ax.plot(o["tokens"], hbar[o["tokens"] - 1], "x", color="k", ms=6)
        shown[key] += 1
    if np.isfinite(TAU_STAR[mk]):
        ax.axhline(TAU_STAR[mk], ls="--", c="k", lw=1, label=r"$\tau^*$")
    ax.axvline(MIN_TOKENS, ls=":", c="grey", lw=1)
    ax.set_title(model_label(mk)); ax.set_xlabel("Generated token index t")
    ax.set_ylabel(r"Smoothed entropy $\bar{H}_L^{(t)}$ (nats)"); ax.legend(fontsize=8)
fig.suptitle("Entropy trajectories of test flows under naive decoding (x = breaker trip point)", y=1.03)
save_fig(fig, "F_entropy_trajectories"); plt.show()

# (b) Distribution of mean entropy per flow, by model and dataset (naive, test split)
fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(MK) * len(DS)), 3.8))
data, labels, pos = [], [], []
for i, mk in enumerate(MK):
    for j, dn in enumerate(DS):
        rows = TEST[TEST["dataset"] == dn]
        data.append([np.mean(NAIVE[mk][s]["entropies"]) for s in rows["sample_id"]])
        pos.append(i * (len(DS) + 1) + j); labels.append(f"{mk}\n{dn}")
bp = ax.boxplot(data, positions=pos, widths=0.7, patch_artist=True, showfliers=False)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for k, patch in enumerate(bp["boxes"]):
    patch.set_facecolor(colors[k % len(DS)]); patch.set_alpha(0.6)
ax.set_xticks(pos); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("Mean token entropy per flow (nats)")
ax.set_title("Token entropy by model size and dataset (naive decoding, test flows)")
save_fig(fig, "F_entropy_by_model_dataset"); plt.show()

# (c) Tau sweep on the calibration split
if TAU_TABLES:
    tt = pd.concat(TAU_TABLES); tt = tt[(tt["window"] == WINDOW) & np.isfinite(tt["tau"])]
    fig, axes = plt.subplots(1, len(MK), figsize=(4.2 * len(MK), 3.4), squeeze=False)
    for ax, mk in zip(axes[0], MK):
        g = tt[tt["model"] == mk]
        ax.plot(g["tau"], g["error"], label="Error rate")
        ax.plot(g["tau"], g["norm_tokens"], label="Tokens / $T_{max}$")
        ax.plot(g["tau"], g["trip_rate"], label="Trip rate", ls=":")
        if np.isfinite(TAU_STAR[mk]):
            ax.axvline(TAU_STAR[mk], c="k", ls="--", lw=1, label=r"$\tau^*$")
        ax.set_title(model_label(mk)); ax.set_xlabel(r"Entropy threshold $\tau$ (nats)")
        ax.set_ylabel("Rate"); ax.legend(fontsize=8)
    fig.suptitle("Threshold sensitivity (calibration split)", y=1.03)
    save_fig(fig, "F_tau_sensitivity"); plt.show()

# (d) Token CDF: naive vs EGCB (both measured)
fig, axes = plt.subplots(1, len(MK), figsize=(4.2 * len(MK), 3.4), squeeze=False)
for ax, mk in zip(axes[0], MK):
    for lab, o in (("Naive", OUT_NAIVE[mk]), ("EGCB", OUT_NSACB[mk])):
        x = np.sort(o["tokens"].values); ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", label=lab)
    ax.set_title(model_label(mk)); ax.set_xlabel("Generated tokens per flow")
    ax.set_ylabel("Cumulative fraction of flows"); ax.legend()
fig.suptitle("Generated-token distribution on test flows", y=1.03)
save_fig(fig, "F_token_cdf"); plt.show()

# (e) Accuracy vs compute for all methods (all test flows)
fig, axes = plt.subplots(1, len(MK), figsize=(4.4 * len(MK), 3.6), squeeze=False)
for ax, mk in zip(axes[0], MK):
    g = table5[(table5["Model"] == mk) & (table5["Dataset"] == "ALL")]
    for (_, r), mkr in zip(g.iterrows(), ["s", "o", "D", "^", "v", "P"]):
        ax.scatter(r["_tokens"], r["_acc"], s=45, marker=mkr, label=r["Method"])
    ax.set_title(model_label(mk)); ax.set_xlabel("Mean generated tokens per flow"); ax.set_ylabel("Accuracy")
handles, labels_ = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.12))
fig.suptitle("Accuracy vs. LLM compute (test flows, all datasets)", y=1.03)
save_fig(fig, "F_accuracy_vs_tokens"); plt.show()

# (f) AUROC heatmaps: any failure vs. wrong answer only (diverging scale centred on chance = 0.5)
panels = [("any_failure (no answer or wrong)", "Any failure\n(no answer or wrong answer)"),
          ("wrong_answer (answered flows only)", "Wrong answer\n(only flows that answered)")]
fig, axes = plt.subplots(1, 2, figsize=(2 * (1.7 * len(DS) + 1.2), 0.75 * len(MK) + 2.0), squeeze=False)
cmap = plt.get_cmap("RdBu").copy(); cmap.set_bad("#d9d9d9")
for ax, (dname, title) in zip(axes[0], panels):
    g = table6[(table6["Failure definition"] == dname) & (table6["Dataset"] != "ALL")]
    piv = g.pivot(index="Model", columns="Dataset", values="_auc_early").reindex(index=MK, columns=DS)
    npiv = g.pivot(index="Model", columns="Dataset", values="n").reindex(index=MK, columns=DS)
    fpiv = g.pivot(index="Model", columns="Dataset", values="Failures (n)").reindex(index=MK, columns=DS)
    im = ax.imshow(np.ma.masked_invalid(piv.values.astype(float)), vmin=0.0, vmax=1.0, cmap=cmap)
    ax.set_xticks(range(len(DS))); ax.set_xticklabels(DS, fontsize=8)
    ax.set_yticks(range(len(MK))); ax.set_yticklabels([model_label(m) for m in MK], fontsize=8)
    for i in range(len(MK)):
        for j in range(len(DS)):
            v = piv.values[i, j]
            txt = "n/a\n(one outcome)" if np.isnan(v) else f"{v:.2f}"
            txt += f"\nn={int(npiv.values[i, j])}, fail={int(fpiv.values[i, j])}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=7, color="black")
    ax.set_title(title, fontsize=9)
cbar = fig.colorbar(im, ax=axes[0].tolist(), shrink=0.9)
cbar.set_label("AUROC of early entropy\n(>0.5: high entropy = failure; 0.5 = chance;\n<0.5: failures are low-entropy)", fontsize=8)
fig.suptitle(f"Does entropy in the first {EARLY_TOKENS} generated tokens predict LLM failure? (test flows)", y=1.04, fontsize=10)
save_fig(fig, "F_entropy_failure_auroc"); plt.show()

In [ ]:
# =====================================================================
# 14. TEXT FOR THE "EXPERIMENTAL SETUP" SUBSECTION (fill-in from real values)
# =====================================================================
prec = {mk: (next(iter(NAIVE[mk].values()))["precision"] if NAIVE[mk] else "n/a") for mk in MODELS}
setup = f"""
Hardware: {ENV.get('gpu')} ({ENV.get('gpu_mem_gb', '-')} GB), {ENV.get('cpu_count')} CPU cores, {ENV.get('ram_gb', '-')} GB RAM.
Software: Python {ENV['python']}, PyTorch {ENV['torch']}, Transformers {ENV['transformers']}, scikit-learn {ENV['sklearn']}, pandas {ENV['pandas']}.
Reflex kernel: Random Forest ({RF_PARAMS}); inputs routed to the LLM when |P(attack|x)-0.5| <= delta = {DELTA}.
Cognitive kernel: {', '.join(f"{mk} = {c['hf_id']} ({prec[mk]})" for mk, c in MODELS.items())}; greedy decoding; chat template of each model.
Prompt (phi): the {FEATURE_TOP_K} most important RF features (training split) listed as 'name: value'{'; the prompt also shows the ' + str(N_REF_PER_CLASS) + ' most similar benign and ' + str(N_REF_PER_CLASS) + ' most similar attack TRAINING flows with their labels (nearest neighbours on signed-log standardised features, random order)' if PROMPT_MODE == 'retrieval' else ' (zero-shot)'}; the model is asked to reason in 2-4 sentences and end with 'Final answer: BENIGN/ATTACK'; its reply is pre-filled with '{ASSISTANT_PREFIX}' so reasoning precedes the answer.
Circuit breaker: token entropy in nats from the full next-token distribution; smoothing window k = {WINDOW}; breaker inactive for the first {MIN_TOKENS} tokens; T_max = {MAX_NEW_TOKENS} generated tokens.
Tau selection: per model, on {N_CALIB} training flows per dataset whose out-of-bag reflex confidence is <= delta (no test flows), minimising J = {LAMBDA}*error + {1-LAMBDA:.1f}*tokens/T_max. Selected: {', '.join(f"{mk}: {v:.3f}" for mk, v in TAU_STAR.items())}.
Fallback: when the breaker trips or no answer is produced, the reflex prediction is returned.
Evaluation flows per dataset (test split): up to {ROUTED_CAP} routed flows + {N_STRESS} balanced stress flows; confidence intervals: {BOOTSTRAP} bootstrap resamples (95%).
Seed: {SEED}.
"""
print(setup)
with open(out("experimental_setup.txt"), "w") as f:
    f.write(setup)

In [ ]:
# =====================================================================
# 15. WHICH OUTPUT GOES WHERE IN THE REVISED PAPER
# =====================================================================
MAPPING = """
tables/T1_datasets.csv                     -> Table 1 (dataset summary; replaces '100000 samples')
tables/T2_reflex_kernel.csv                -> new table: reflex kernel performance, latency, routing
tables/T2b_delta_sweep.csv + F_reflex_routing_vs_delta  -> delta sensitivity
tables/T3_main_results_naive_vs_nsacb.csv  -> replaces Tables 2, 3 and 6 (measured tokens and latency)
tables/T4_tau_sweep_calibration.csv + F_tau_sensitivity -> replaces Table 4
tables/T5_ablation_baselines.csv + F_accuracy_vs_tokens -> Section 5 ablation / comparison with baselines
tables/T6_entropy_predicts_failures_auroc.csv + F_entropy_failure_auroc -> evidence for (or against) 'entropy signals unstable reasoning' (use the wrong_answer rows for that claim)
tables/T7_cascade_end_to_end.csv           -> end-to-end system result (replaces Table 5 and 'reflex 0.01 s')
tables/T8_live_vs_replay_agreement.csv     -> supplementary: live breaker matches replay
figures/F_entropy_by_model_dataset         -> replaces Figs. 2, 3, 4
figures/F_token_cdf                        -> replaces Fig. 6
figures/F_entropy_trajectories             -> new: shows the breaker in action
experimental_setup.txt                     -> text for the new Experimental Setup subsection
llm_runs/*.jsonl                           -> raw per-flow logs (release with the code)
"""
print(MAPPING)
print("All outputs are in", RESULTS_DIR)

In [ ]:
# =====================================================================
# 16. OFFLINE ANALYSES ON THE SAVED LOGS  (no new LLM generations; CPU is fine)
#   A. Oracle upper bound for the entropy breaker: tau chosen ON THE TEST FLOWS
#      (not deployable; shows the best any entropy threshold could have done),
#      compared with a random breaker that falls back on the same number of flows.
#   B. Answer-level verifier: after the LLM answers, fall back to the Random Forest
#      if the entropy of the ANSWER WORD (ATTACK/BENIGN) is high. Threshold chosen on
#      calibration (training) flows = deployable; also on test flows = oracle.
#   C. Headroom: per-flow oracle that always picks whichever of LLM / RF is correct.
# =====================================================================
import os, re, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

RES = globals().get("RESULTS_DIR", "/content/drive/MyDrive/egcb_results_retrieval")
CFG = json.load(open(os.path.join(RES, "config.json")))
WIN, MINT = CFG["WINDOW"], CFG["MIN_TOKENS"]
LAM, SEED_, B_ = CFG["LAMBDA"], CFG["SEED"], CFG["BOOTSTRAP"]
TMAX = CFG["MAX_NEW_TOKENS"]
TAU_STAR_SAVED = json.load(open(os.path.join(RES, "tau_star.json")))["tau_star"]
os.makedirs(os.path.join(RES, "tables"), exist_ok=True); os.makedirs(os.path.join(RES, "figures"), exist_ok=True)

EV = pd.read_csv(os.path.join(RES, "data", "eval_set.csv"))
EV_T = EV[EV["split"] == "test"].reset_index(drop=True)
EV_C = EV[EV["split"] == "calib"].reset_index(drop=True)
t2 = pd.read_csv(os.path.join(RES, "tables", "T2_reflex_kernel.csv"))
RLAT = dict(zip(t2["Dataset"], t2["Median latency per flow (ms)"] / 1000))

def load_naive(mk):
    recs = [json.loads(l) for l in open(os.path.join(RES, "llm_runs", f"{mk}__naive.jsonl")) if l.strip()]
    ver = pd.Series([r.get("run_version") for r in recs]).value_counts().index[0]   # the version used for the paper
    return {r["sample_id"]: r for r in recs if r.get("run_version") == ver}

MKS = [mk for mk in ["small", "medium", "large", "xlarge"] if os.path.exists(os.path.join(RES, "llm_runs", f"{mk}__naive.jsonl"))]
MODELS_ = CFG["MODELS"]
NV = {mk: load_naive(mk) for mk in MKS}
print("Models:", MKS, "| test flows:", len(EV_T), "| calibration flows:", len(EV_C))

def mlabel(mk):
    return f"{mk} ({MODELS_[mk]['params_b']:.1f}B)"

def boot_ci(v, B=B_, seed=SEED_):
    v = np.asarray(v, float)
    bs = v[np.random.default_rng(seed).integers(0, len(v), (B, len(v)))].mean(1)
    return v.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)

def cis(v, scale=100, nd=1):
    m, lo, hi = boot_ci(v)
    return f"{m*scale:.{nd}f} [{lo*scale:.{nd}f}, {hi*scale:.{nd}f}]"

def auc_ci(y, s, B=B_, seed=SEED_):
    y, s = np.asarray(y), np.asarray(s, float)
    ok = np.isfinite(s); y, s = y[ok], s[ok]
    if len(y) == 0 or len(np.unique(y)) < 2:
        return "n/a (one outcome)", np.nan
    rng = np.random.default_rng(seed); bs = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            bs.append(roc_auc_score(y[i], s[i]))
    a = roc_auc_score(y, s)
    return f"{a:.3f} [{np.percentile(bs, 2.5):.3f}, {np.percentile(bs, 97.5):.3f}]", a

# ---------- breaker replay (identical to the main notebook) ----------
def smoothed_eligible(rec, window=WIN, min_tokens=MINT):
    H = np.asarray(rec["entropies"], float); n = len(H)
    c = np.concatenate([[0.0], np.cumsum(H)]); t = np.arange(1, n + 1)
    lo = np.maximum(0, t - window); hbar = (c[t] - c[lo]) / (t - lo)
    elig = t >= min_tokens
    if rec["answer_step"]:
        elig &= t != rec["answer_step"]
    if rec["stop_reason"] == "eos":
        elig &= t != n
    return np.where(elig, hbar, -np.inf)

def first_trip(hb, tau):
    if not np.isfinite(tau):
        return None
    idx = np.flatnonzero(hb > tau)
    return int(idx[0]) + 1 if idx.size else None

# ---------- per-flow table ----------
ANSWER_RE = re.compile(r"final\s+answer\s*[:\-]?\s*\**\s*(benign|attack|malicious|normal)", re.I)
try:
    from transformers import AutoTokenizer
    TOK = AutoTokenizer.from_pretrained(MODELS_[MKS[0]]["hf_id"])   # all Qwen2.5 sizes share one tokenizer
except Exception as e:
    TOK = None
    print("Tokenizer unavailable (", e, ") -> using the alignment-free answer signal only")

def answer_token_index(rec):
    """0-based index of the first token of the answer word, or None."""
    if TOK is None or not rec["answer_step"]:
        return None
    m = ANSWER_RE.search(rec["text"])
    if not m:
        return None
    prefix = rec["text"][: m.start(1)].rstrip(" ")
    c = len(TOK(prefix, add_special_tokens=False)["input_ids"])
    return int(min(c, rec["answer_step"] - 1, len(rec["entropies"]) - 1))

def flow_rows(ev, mk):
    rows, aligned = [], 0
    for r in ev.itertuples(index=False):
        rec = NV[mk][r.sample_id]; H = np.asarray(rec["entropies"], float)
        ans = rec["llm_answer"]
        llm_pred = None if ans is None else int(ans == "attack")
        ai = answer_token_index(rec)
        aligned += ai is not None
        n = len(H)
        last4 = float(H[max(0, (rec["answer_step"] or n) - 4):(rec["answer_step"] or n)].max()) if ans else np.nan
        rows.append(dict(
            sample_id=r.sample_id, dataset=r.dataset, subset=r.subset, y=int(r.y), rf_pred=int(r.y_reflex),
            llm_pred=llm_pred, answered=ans is not None,
            sys_pred=int(r.y_reflex) if llm_pred is None else llm_pred,          # naive system (RF if no answer)
            ent_early=float(H[:32].mean()), ent_full=float(H.mean()),
            ent_answer=float(H[ai]) if ai is not None else np.nan,
            ent_answer_pm1=float(H[max(0, ai - 1): ai + 2].max()) if ai is not None else np.nan,
            ent_last4=last4, tokens=rec["gen_tokens"], latency=rec["total_s"] + RLAT[r.dataset],
            hb=smoothed_eligible(rec), t_cum=rec["t_cum"]))
    df = pd.DataFrame(rows)
    df["rf_ok"] = (df["rf_pred"] == df["y"]).astype(int)
    df["sys_ok"] = (df["sys_pred"] == df["y"]).astype(int)
    df["llm_wrong"] = np.where(df["answered"], (df["llm_pred"] != df["y"]).astype(float), np.nan)
    return df, aligned

FT, FC = {}, {}
for mk in MKS:
    FT[mk], a1 = flow_rows(EV_T, mk); FC[mk], a2 = flow_rows(EV_C, mk)
    print(f"{mk}: answer word located by tokenizer in {a1}/{int(FT[mk]['answered'].sum())} answered test flows")

# =====================================================================
# A. ORACLE TAU vs RANDOM BREAKER
# =====================================================================
def breaker_eval(df, tau):
    ok, tok, lat, trips = [], [], [], 0
    for r in df.itertuples(index=False):
        t = first_trip(r.hb, tau)
        if t is None:
            ok.append(r.sys_ok); tok.append(r.tokens); lat.append(r.latency)
        else:
            trips += 1; ok.append(r.rf_ok); tok.append(t); lat.append(r.t_cum[t - 1] + (r.latency - r.t_cum[-1]))
    return np.array(ok), np.array(tok), np.array(lat), trips / len(df)

rowsA, curves = [], {}
for mk in MKS:
    for subset in ["routed", "all"]:
        df = FT[mk] if subset == "all" else FT[mk][FT[mk]["subset"] == "routed"]
        vals = np.concatenate([h[np.isfinite(h)] for h in df["hb"]])
        grid = np.append(np.unique(np.quantile(vals, np.linspace(0, 1, 201))), np.inf)
        acc_rf, acc_sys = df["rf_ok"].mean(), df["sys_ok"].mean()
        res = []
        for tau in grid:
            ok, tok, lat, q = breaker_eval(df, tau)
            res.append(dict(tau=tau, acc=ok.mean(), tokens=tok.mean(), trip=q,
                            rand_acc=(1 - q) * acc_sys + q * acc_rf,       # random fallback on the same share of flows
                            J=LAM * (1 - ok.mean()) + (1 - LAM) * tok.mean() / TMAX))
        res = pd.DataFrame(res); curves[(mk, subset)] = res
        best_acc = res.loc[res["acc"] == res["acc"].max()].iloc[-1]        # ties -> largest tau (least intervention)
        best_J = res.loc[res["J"] == res["J"].min()].iloc[-1]
        res["gain_vs_random"] = res["acc"] - res["rand_acc"]
        max_gain = round(100 * res["gain_vs_random"].max(), 2)
        policies = [("Reflex only (RF)", None), ("LLM naive (RF if no answer)", np.inf),
                    ("EGCB, tau* from calibration (deployed)", TAU_STAR_SAVED[mk]),
                    ("ORACLE tau, max accuracy on test", best_acc["tau"]),
                    ("ORACLE tau, min J on test", best_J["tau"])]
        for name, tau in policies:
            if tau is None:
                ok, tok, lat, q = df["rf_ok"].values, np.zeros(len(df)), np.array([RLAT[d] for d in df["dataset"]]), 1.0
                rand = np.nan
            else:
                ok, tok, lat, q = breaker_eval(df, tau)
                rand = (1 - q) * acc_sys + q * acc_rf
            rowsA.append({"Model": mk, "Test flows": subset, "Policy": name,
                          "tau": "-" if tau is None else ("off" if not np.isfinite(tau) else round(float(tau), 4)),
                          "n": len(df), "Accuracy % [95% CI]": cis(ok), "Breaker trips %": round(100 * q, 1),
                          "Random breaker, same trip rate: accuracy %": "-" if np.isnan(rand) else round(100 * rand, 1),
                          "Gain over random (pts)": "-" if np.isnan(rand) else round(100 * (ok.mean() - rand), 2),
                          "Mean generated tokens": round(tok.mean(), 1), "Mean latency s": round(lat.mean(), 3),
                          "Best gain over random at any tau (pts)": max_gain})
T9 = pd.DataFrame(rowsA)
T9.to_csv(os.path.join(RES, "tables", "T9_oracle_tau_vs_random.csv"), index=False)
print("[saved] tables/T9_oracle_tau_vs_random.csv")

fig, axes = plt.subplots(1, len(MKS), figsize=(4.3 * len(MKS), 3.6), squeeze=False)
for ax, mk in zip(axes[0], MKS):
    res = curves[(mk, "routed")].sort_values("trip")
    ax.plot(100 * res["trip"], 100 * res["acc"], label="Entropy breaker (all tau)", lw=1.8)
    ax.plot(100 * res["trip"], 100 * res["rand_acc"], "--", label="Random breaker, same trip rate", lw=1.4)
    ax.set_title(mlabel(mk)); ax.set_xlabel("Routed flows where the breaker trips (%)")
    ax.set_ylabel("Accuracy on routed test flows (%)"); ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=8)
fig.suptitle("Entropy breaker vs. random breaker on routed test flows (every tau; oracle view)", y=1.03)
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RES, "figures", f"F_breaker_vs_random.{ext}"), dpi=300, bbox_inches="tight")
plt.show()

# =====================================================================
# B. ANSWER-LEVEL VERIFIER + SIGNAL AUROC
# =====================================================================
SIGNALS = {"ent_early": "mean entropy, first 32 tokens", "ent_full": "mean entropy, whole output",
           "ent_answer": "entropy of the answer word (tokenizer-aligned)",
           "ent_answer_pm1": "max entropy, answer word +/- 1 token", "ent_last4": "max entropy, last 4 tokens"}
rowsS = []
for mk in MKS:
    for ds in sorted(EV_T["dataset"].unique()) + ["ALL"]:
        df = FT[mk] if ds == "ALL" else FT[mk][FT[mk]["dataset"] == ds]
        d = df[df["answered"]]
        row = {"Model": mk, "Dataset": ds, "Answered flows": len(d), "Wrong answers": int(d["llm_wrong"].sum())}
        for s, name in SIGNALS.items():
            row[f"AUROC {name}"] = auc_ci(d["llm_wrong"].values, d[s].values)[0]
        rowsS.append(row)
T10 = pd.DataFrame(rowsS)
T10.to_csv(os.path.join(RES, "tables", "T10_answer_signal_auroc.csv"), index=False)
print("[saved] tables/T10_answer_signal_auroc.csv")

def verifier_ok(df, sig, theta):
    use_llm = df["answered"].values & (df[sig].fillna(np.inf).values <= theta)
    ok = np.where(use_llm, df["llm_pred"].fillna(-1).values == df["y"].values, df["rf_ok"].values).astype(int)
    return ok, 1 - use_llm.mean()

rowsV = []
VERIFIER_SIGNAL = "ent_answer" if TOK is not None else "ent_last4"
for mk in MKS:
    cal = FC[mk]
    vals = cal.loc[cal["answered"], VERIFIER_SIGNAL].dropna().values
    grid = np.append(np.unique(np.quantile(vals, np.linspace(0, 1, 201))) if len(vals) else [], [-np.inf, np.inf])
    accs = [(verifier_ok(cal, VERIFIER_SIGNAL, th)[0].mean(), th) for th in grid]
    best_cal = max(a for a, _ in accs); theta_cal = max(th for a, th in accs if a == best_cal)
    for subset in ["routed", "all"]:
        df = FT[mk] if subset == "all" else FT[mk][FT[mk]["subset"] == "routed"]
        tv = df.loc[df["answered"], VERIFIER_SIGNAL].dropna().values
        tgrid = np.append(np.unique(np.quantile(tv, np.linspace(0, 1, 201))) if len(tv) else [], [-np.inf, np.inf])
        taccs = [(verifier_ok(df, VERIFIER_SIGNAL, th)[0].mean(), th) for th in tgrid]
        best_t = max(a for a, _ in taccs); theta_or = max(th for a, th in taccs if a == best_t)
        either = ((df["rf_ok"] == 1) | ((df["answered"]) & (df["llm_pred"] == df["y"]))).astype(int).values
        entries = [("Reflex only (RF)", df["rf_ok"].values, 1.0, "-"),
                   ("LLM naive (RF if no answer)", df["sys_ok"].values, 1 - df["answered"].mean(), "-")]
        ok, fb = verifier_ok(df, VERIFIER_SIGNAL, theta_cal); entries.append(("Verifier, theta from calibration (deployable)", ok, fb, theta_cal))
        ok, fb = verifier_ok(df, VERIFIER_SIGNAL, theta_or); entries.append(("ORACLE verifier, theta on test", ok, fb, theta_or))
        entries.append(("ORACLE per-flow (LLM or RF, whichever is right) - upper bound", either, np.nan, "-"))
        for name, ok, fb, th in entries:
            rowsV.append({"Model": mk, "Test flows": subset, "Policy": name, "Signal": SIGNALS[VERIFIER_SIGNAL],
                          "theta": th if isinstance(th, str) else ("never trust LLM" if th == -np.inf else ("always trust LLM" if th == np.inf else round(float(th), 4))),
                          "n": len(df), "Accuracy % [95% CI]": cis(ok),
                          "Falls back to RF %": "-" if np.isnan(fb) else round(100 * fb, 1)})
T11 = pd.DataFrame(rowsV)
T11.to_csv(os.path.join(RES, "tables", "T11_answer_level_verifier.csv"), index=False)
print("[saved] tables/T11_answer_level_verifier.csv")

fig, axes = plt.subplots(1, len(MKS), figsize=(4.3 * len(MKS), 3.6), squeeze=False)
for ax, mk in zip(axes[0], MKS):
    df = FT[mk][FT[mk]["subset"] == "routed"]
    tv = df.loc[df["answered"], VERIFIER_SIGNAL].dropna().values
    ths = np.unique(np.append(np.quantile(tv, np.linspace(0, 1, 101)), [-np.inf, np.inf])) if len(tv) else np.array([-np.inf, np.inf])
    pts = sorted((100 * verifier_ok(df, VERIFIER_SIGNAL, th)[1], 100 * verifier_ok(df, VERIFIER_SIGNAL, th)[0].mean()) for th in ths)
    ax.plot([p[0] for p in pts], [p[1] for p in pts], lw=1.8, label="Answer-entropy verifier (all thresholds)")
    q0, q1 = 100 * (1 - df["answered"].mean()), 100.0
    a0, a1 = 100 * df["sys_ok"].mean(), 100 * df["rf_ok"].mean()
    ax.plot([q0, q1], [a0, a1], "--", lw=1.4, label="Random fallback (same rate)")
    ax.set_title(mlabel(mk)); ax.set_xlabel("Routed flows sent back to the RF (%)")
    ax.set_ylabel("Accuracy on routed test flows (%)"); ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=8)
fig.suptitle("Answer-level verifier on routed test flows (every threshold; oracle view)", y=1.03)
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RES, "figures", f"F_answer_verifier.{ext}"), dpi=300, bbox_inches="tight")
plt.show()

# ---------- compact printout to paste back ----------
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("\n=== T9 oracle tau vs random (routed test flows)\n", T9[T9["Test flows"] == "routed"].to_string(index=False))
print("\n=== T10 AUROC of each signal for WRONG answers (answered test flows)\n", T10.to_string(index=False))
print("\n=== T11 answer-level verifier (routed test flows)\n", T11[T11["Test flows"] == "routed"].to_string(index=False))

In [ ]:
# =====================================================================
# 17. NEURAL NETWORK REFLEX  (answers Reviewer 2, point 11)
#     Trains a Multi-Layer Perceptron (MLP) reflex on the same splits and
#     compares it with the Random Forest as the cheap model of the cascade.
#     CPU only, a few minutes. No LLM calls: the logged generations are reused.
#
#     Run after cell 5 (the reflex kernel) in the same session, or run
#     cells 0-6 first if the runtime was restarted.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

for name in ["DATA", "EVAL", "RESULTS_DIR", "DELTA", "SEED", "REFLEX_LAT"]:
    if name not in globals():
        raise RuntimeError(f"{name} is missing - run cells 0 to 6 of the notebook first")

MLP_PARAMS = dict(hidden_layer_sizes=(128, 64), activation="relu", alpha=1e-4,
                  batch_size=512, learning_rate_init=1e-3, max_iter=60,
                  early_stopping=True, n_iter_no_change=5, validation_fraction=0.1,
                  random_state=SEED)

def metrics(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(acc=accuracy_score(y, p), f1=f1_score(y, p, zero_division=0),
                fnr=fn / (tp + fn) if (tp + fn) else np.nan,
                fpr=fp / (tn + fp) if (tn + fp) else np.nan)

# ---------------------------------------------------------------- train
rows, MLP = [], {}
for name, ds in DATA.items():
    tr, te = ds["train"], ds["test"]
    Xtr, Xte = make_X(ds, tr), make_X(ds, te)
    ytr, yte = tr["y"].values, te["y"].values
    clf = make_pipeline(StandardScaler(), MLPClassifier(**MLP_PARAMS))
    t0 = time.perf_counter(); clf.fit(Xtr, ytr); fit_s = time.perf_counter() - t0
    proba = clf.predict_proba(Xte)[:, 1]
    te["p_mlp"] = proba
    te["u_mlp"] = np.abs(proba - 0.5)
    te["y_mlp"] = (proba >= 0.5).astype(int)
    MLP[name] = clf

    lat = []
    rng = np.random.default_rng(SEED)
    for i in rng.integers(0, len(Xte), 200):
        t0 = time.perf_counter(); clf.predict_proba(Xte[i:i + 1]); lat.append(time.perf_counter() - t0)
    lat_ms = 1000 * float(np.median(lat))

    for label, pred, u in [("Random Forest", te["y_reflex"].values, te["u"].values),
                           ("Neural Network (MLP)", te["y_mlp"].values, te["u_mlp"].values)]:
        m = metrics(yte, pred)
        routed = u <= DELTA
        rows.append({"Dataset": name, "Reflex model": label,
                     "Accuracy (%)": round(100 * m["acc"], 2), "F1 (%)": round(100 * m["f1"], 2),
                     "FNR (%)": round(100 * m["fnr"], 2), "FPR (%)": round(100 * m["fpr"], 2),
                     "Routed (%)": round(100 * routed.mean(), 3), "Routed (n)": int(routed.sum()),
                     "Accuracy on routed flows (%)": round(100 * accuracy_score(yte[routed], pred[routed]), 1) if routed.any() else np.nan,
                     "Latency per flow (ms)": round(lat_ms if label.startswith("Neural") else 1000 * REFLEX_LAT[name], 2),
                     "Training time (s)": round(fit_s if label.startswith("Neural") else np.nan, 1)})
    agree = (te["y_reflex"].values == te["y_mlp"].values).mean()
    both_routed = ((te["u"].values <= DELTA) & (te["u_mlp"].values <= DELTA)).sum()
    print(f"{name}: MLP acc={metrics(yte, te['y_mlp'].values)['acc']:.4f} | RF acc={metrics(yte, te['y_reflex'].values)['acc']:.4f} | "
          f"predictions agree on {100*agree:.2f}% of flows | flows routed by both: {both_routed}")

T12 = pd.DataFrame(rows)
T12.to_csv(os.path.join(RESULTS_DIR, "tables", "T12_reflex_rf_vs_mlp.csv"), index=False)
print("[saved] tables/T12_reflex_rf_vs_mlp.csv")

# ------------------------------------------- MLP as the fallback of the cascade
# The evaluated flows were selected by the Random Forest, so this compares the two
# reflexes as FALLBACK models on identical LLM generations; it is not a re-routing.
if "OUT_NAIVE" in globals() and "TEST" in globals():
    mlp_pred = {}
    for name, ds in DATA.items():
        te = ds["test"]
        mlp_pred.update({f"{name}#{i}": int(v) for i, v in zip(te.index, te["y_mlp"].values)})
    rows2 = []
    for mk in MODELS:
        sub = TEST[TEST["subset"] == "routed"]
        y = sub["y"].values
        rf_fb = sub["y_reflex"].values
        mlp_fb = np.array([mlp_pred[s] for s in sub["sample_id"]])
        recs = NAIVE[mk]
        ans = np.array([recs[s]["llm_answer"] for s in sub["sample_id"]], dtype=object)
        llm = np.array([np.nan if a is None else float(a == "attack") for a in ans])
        for fb_name, fb in [("Random Forest", rf_fb), ("Neural Network (MLP)", mlp_fb)]:
            reflex_only = accuracy_score(y, fb)
            pred = np.where(np.isnan(llm), fb, np.nan_to_num(llm))
            naive = accuracy_score(y, pred.astype(int))
            rows2.append({"Model": mk, "Reflex model": fb_name,
                          "Reflex only, routed flows (%)": round(100 * reflex_only, 1),
                          "Naive LLM with this fallback (%)": round(100 * naive, 1),
                          "EGCB (always falls back) (%)": round(100 * reflex_only, 1)})
    T13 = pd.DataFrame(rows2)
    T13.to_csv(os.path.join(RESULTS_DIR, "tables", "T13_cascade_rf_vs_mlp_fallback.csv"), index=False)
    print("[saved] tables/T13_cascade_rf_vs_mlp_fallback.csv")
else:
    T13 = None
    print("Skipped the cascade comparison: run cells 9-12 first so that NAIVE and TEST exist.")

# ---------------------------------------------------------------- figure
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ds_names = list(DATA)
x = np.arange(len(ds_names)); wdt = 0.38
for ax, col, ylab in [(axes[0], "Accuracy (%)", "Test accuracy (%)"),
                      (axes[1], "Routed (%)", "Flows routed to the LLM (%)")]:
    for j, model in enumerate(["Random Forest", "Neural Network (MLP)"]):
        v = [T12[(T12["Dataset"] == d) & (T12["Reflex model"] == model)][col].iloc[0] for d in ds_names]
        ax.bar(x + (j - 0.5) * wdt, v, wdt, label=model)
    ax.set_xticks(x); ax.set_xticklabels(ds_names, fontsize=8)
    ax.set_ylabel(ylab); ax.grid(alpha=0.3, axis="y")
    if col == "Routed (%)":
        ax.set_yscale("log")
axes[0].legend(fontsize=8)
fig.suptitle("Random Forest and Neural Network reflex on the same splits", y=1.02)
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RESULTS_DIR, "figures", f"F_reflex_rf_vs_mlp.{ext}"), dpi=300, bbox_inches="tight")
plt.show()

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
print("\n=== T12 reflex comparison\n", T12.to_string(index=False))
if T13 is not None:
    print("\n=== T13 same LLM generations, different fallback model\n", T13.to_string(index=False))

In [ ]:
# =====================================================================
# 18. SELF-CONSISTENCY BASELINE  (answers Reviewer 2, point 10)
#     An existing alternative to entropy-based control: sample several answers
#     per flow, take the majority vote, and fall back to the reflex when the
#     samples disagree. This tests whether agreement across samples succeeds
#     where token entropy failed.
#
#     Needs the LLM engine of cell 7 (load_llm, encode_prompt, ANSWER_RE) and
#     the evaluation set of cell 6. Resumable: each finished flow is saved.
#
#     Cost: SC_SAMPLES answers per flow for the flows of SC_MODELS. The cell
#     prints an estimate before it starts.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score

SC_MODELS  = ["large", "xlarge"]   # the models worth running; add "medium" if you have time
SC_SAMPLES = 5                     # answers sampled per flow
SC_TEMP    = 0.7
SC_TOP_P   = 0.9
SC_MAX_NEW = 256                   # same budget as the naive runs
SC_SUBSETS = ["routed"]            # test flows to evaluate; calibration flows are always included

for name in ["EVAL", "MODELS", "RESULTS_DIR", "SEED", "REFLEX_LAT", "load_llm", "unload_llm", "encode_prompt", "ANSWER_RE", "ANSWER_MAP"]:
    if name not in globals():
        raise RuntimeError(f"{name} is missing - run cells 0 to 7 of the notebook first")
os.makedirs(os.path.join(RESULTS_DIR, "sc_runs"), exist_ok=True)

SC_FLOWS = EVAL[((EVAL["split"] == "test") & (EVAL["subset"].isin(SC_SUBSETS))) | (EVAL["split"] == "calib")].reset_index(drop=True)
print(f"Flows per model: {len(SC_FLOWS)} ({(SC_FLOWS['split']=='calib').sum()} calibration, {(SC_FLOWS['split']=='test').sum()} test)")
print(f"Rough cost per model: {len(SC_FLOWS)*SC_SAMPLES} generations of up to {SC_MAX_NEW} tokens")

@torch.no_grad()
def sample_answers(bundle, user_text, n=SC_SAMPLES, temperature=SC_TEMP, top_p=SC_TOP_P,
                   max_new_tokens=SC_MAX_NEW, seed=0):
    """Generates n answers for one prompt in a single batch (all rows share the prompt)."""
    tok, model = bundle["tok"], bundle["model"]
    dev = next(model.parameters()).device
    ids = encode_prompt(tok, user_text).to(dev).repeat(n, 1)
    gen = [[] for _ in range(n)]
    done = torch.zeros(n, dtype=torch.bool, device=dev)
    answers = [None] * n
    g = torch.Generator(device=dev.type); g.manual_seed(seed)

    t0 = time.perf_counter()
    out = model(input_ids=ids, use_cache=True)
    past, logits = out.past_key_values, out.logits[:, -1, :]
    for t in range(1, max_new_tokens + 1):
        lf = logits.float() / max(temperature, 1e-6)
        probs = torch.softmax(lf, dim=-1)
        if top_p < 1.0:                                    # nucleus filtering
            sp, si = torch.sort(probs, descending=True, dim=-1)
            keep = (torch.cumsum(sp, dim=-1) - sp) < top_p
            sp = torch.where(keep, sp, torch.zeros_like(sp))
            sp = sp / sp.sum(dim=-1, keepdim=True)
            pick = torch.multinomial(sp, 1, generator=g)
            nxt = si.gather(-1, pick)
        else:
            nxt = torch.multinomial(probs, 1, generator=g)
        nxt = torch.where(done.unsqueeze(1), torch.full_like(nxt, tok.eos_token_id or 0), nxt)
        for i in range(n):
            if done[i]:
                continue
            tid = int(nxt[i, 0])
            if tid in bundle["eos"]:
                done[i] = True; continue
            gen[i].append(tid)
            m = ANSWER_RE.search(tok.decode(gen[i], skip_special_tokens=True))
            if m:
                answers[i] = ANSWER_MAP[m.group(1).lower()]; done[i] = True
        if bool(done.all()) or t == max_new_tokens:
            break
        out = model(input_ids=nxt, past_key_values=past, use_cache=True)
        past, logits = out.past_key_values, out.logits[:, -1, :]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    # latency is measured for the whole batch of n samples, which is how self-consistency
    # would actually be served, and is therefore comparable with the single greedy run
    return dict(answers=answers, tokens=int(sum(len(x) for x in gen)),
                latency=time.perf_counter() - t0,
                texts=[tok.decode(x, skip_special_tokens=True)[:400] for x in gen])

def sc_path(mk):
    return os.path.join(RESULTS_DIR, "sc_runs", f"{mk}__selfconsistency.jsonl")

def load_sc(mk):
    recs = {}
    if os.path.exists(sc_path(mk)):
        with open(sc_path(mk)) as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    if r.get("n") == SC_SAMPLES and r.get("temperature") == SC_TEMP:
                        recs[r["sample_id"]] = r
    return recs

for mk in SC_MODELS:
    done = load_sc(mk)
    todo = SC_FLOWS[~SC_FLOWS["sample_id"].isin(done)]
    if todo.empty:
        print(f"[{mk}] all {len(SC_FLOWS)} flows already sampled"); continue
    print(f"[{mk}] sampling {len(todo)} flows x {SC_SAMPLES} answers")
    bundle = load_llm(MODELS[mk]); t_start = time.time()
    with open(sc_path(mk), "a") as f:
        for j, r in enumerate(todo.itertuples(index=False), 1):
            rec = sample_answers(bundle, r.prompt, seed=SEED + abs(hash(r.sample_id)) % 100000)
            rec.update(sample_id=r.sample_id, dataset=r.dataset, subset=r.subset, split=r.split,
                       model=mk, n=SC_SAMPLES, temperature=SC_TEMP, top_p=SC_TOP_P)
            f.write(json.dumps(rec) + "\n"); f.flush()
            if j % 10 == 0 or j == len(todo):
                el = time.time() - t_start
                print(f"  {j}/{len(todo)}  elapsed {el/60:.1f} min  ETA {el/j*(len(todo)-j)/60:.1f} min")
    unload_llm(bundle)

# ---------------------------------------------------------------- evaluation
def vote(rec):
    """Majority label and the share of samples that agree with it."""
    labs = [a for a in rec["answers"] if a is not None]
    if not labs:
        return None, 0.0
    att = labs.count("attack"); ben = labs.count("benign")
    lab = "attack" if att >= ben else "benign"
    return lab, max(att, ben) / len(rec["answers"])

rows = []
for mk in SC_MODELS:
    recs = load_sc(mk)
    if not recs:
        continue
    ev = SC_FLOWS[SC_FLOWS["sample_id"].isin(recs)]
    cal = ev[ev["split"] == "calib"]; tst = ev[ev["split"] == "test"]

    def outcome(sub, thr):
        y, pred = sub["y"].values, []
        for r in sub.itertuples(index=False):
            lab, agree = vote(recs[r.sample_id])
            pred.append(int(r.y_reflex) if (lab is None or agree < thr) else int(lab == "attack"))
        return np.array(pred), y

    grid = sorted({round(k / SC_SAMPLES, 3) for k in range(1, SC_SAMPLES + 1)})
    def cal_acc(th):
        pred, y = outcome(cal, th)
        return accuracy_score(y, pred)
    best = max(grid, key=cal_acc) if len(cal) else grid[0]

    for sub_name, sub in [("routed test flows", tst)]:
        if len(sub) == 0:
            continue
        maj, y = outcome(sub, 0.0)                    # always trust the majority vote
        cal_p, _ = outcome(sub, best)                 # abstain when agreement is low
        toks = np.mean([recs[s]["tokens"] for s in sub["sample_id"]])
        lat = np.mean([recs[s]["latency"] + REFLEX_LAT[d] for s, d in zip(sub["sample_id"], sub["dataset"])])
        agr = np.mean([vote(recs[s])[1] for s in sub["sample_id"]])
        naive = None
        if "NAIVE" in globals() and mk in NAIVE:
            nv = [NAIVE[mk][s] for s in sub["sample_id"]]
            nv_pred = [int(r.y_reflex) if rec["llm_answer"] is None else int(rec["llm_answer"] == "attack")
                       for rec, r in zip(nv, sub.itertuples(index=False))]
            naive = round(100 * accuracy_score(y, nv_pred), 1)
        rows.append({"Model": mk, "Flows": f"{sub_name} (n={len(sub)})",
                     "Reflex only (%)": round(100 * accuracy_score(y, sub["y_reflex"].values), 1),
                     "Naive LLM, greedy (%)": naive,
                     "Self-consistency, majority vote (%)": round(100 * accuracy_score(y, maj), 1),
                     f"Self-consistency + fallback when agreement < {best} (%)": round(100 * accuracy_score(y, cal_p), 1),
                     "Mean agreement": round(float(agr), 3),
                     "Tokens per flow": round(float(toks), 1), "Latency per flow (s)": round(float(lat), 2)})

T14 = pd.DataFrame(rows)
if len(T14):
    T14.to_csv(os.path.join(RESULTS_DIR, "tables", "T14_self_consistency.csv"), index=False)
    print("[saved] tables/T14_self_consistency.csv")
    pd.set_option("display.width", 220); pd.set_option("display.max_columns", 20)
    print("\n=== T14 self-consistency versus greedy decoding and the reflex\n", T14.to_string(index=False))
else:
    print("No sampled flows yet.")

In [ ]:
# =====================================================================
# 19. SELF-CONSISTENCY: CONFIDENCE INTERVALS AND AGREEMENT AS A SIGNAL
#     Reads the samples saved by cell 18. No GPU, runs in seconds.
#     Answers: (a) is the majority vote better than greedy decoding?
#              (b) does agreement across samples predict a wrong answer,
#                  where token entropy did not?
# =====================================================================
import os, json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

RES = globals().get("RESULTS_DIR", "/content/drive/MyDrive/egcb_results_retrieval")
CFG = json.load(open(os.path.join(RES, "config.json")))
SEED_, B_ = CFG["SEED"], CFG["BOOTSTRAP"]
EV = pd.read_csv(os.path.join(RES, "data", "eval_set.csv"))
Y = dict(zip(EV.sample_id, EV.y)); YR = dict(zip(EV.sample_id, EV.y_reflex))
DS = dict(zip(EV.sample_id, EV.dataset)); SPLIT = dict(zip(EV.sample_id, EV.split))
t2 = pd.read_csv(os.path.join(RES, "tables", "T2_reflex_kernel.csv"))
RLAT = dict(zip(t2["Dataset"], t2["Median latency per flow (ms)"] / 1000))

def boot_ci(v, B=B_, seed=SEED_):
    v = np.asarray(v, float)
    if len(v) == 0:
        return (np.nan, np.nan, np.nan)
    bs = v[np.random.default_rng(seed).integers(0, len(v), (B, len(v)))].mean(1)
    return v.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)

def pct(v, nd=1):
    m, lo, hi = boot_ci(v)
    return f"{100*m:.{nd}f} [{100*lo:.{nd}f}, {100*hi:.{nd}f}]"

def auc_ci(y, s, B=B_, seed=SEED_):
    y, s = np.asarray(y), np.asarray(s, float)
    if len(np.unique(y)) < 2:
        return "n/a (one outcome)"
    rng = np.random.default_rng(seed); bs = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            bs.append(roc_auc_score(y[i], s[i]))
    return f"{roc_auc_score(y, s):.3f} [{np.percentile(bs,2.5):.3f}, {np.percentile(bs,97.5):.3f}]"

def vote(rec):
    labs = [a for a in rec["answers"] if a is not None]
    if not labs:
        return None, 0.0
    att, ben = labs.count("attack"), labs.count("benign")
    return ("attack" if att >= ben else "benign"), max(att, ben) / len(rec["answers"])

sc_files = sorted(f for f in os.listdir(os.path.join(RES, "sc_runs")) if f.endswith(".jsonl"))
MKS = [f.split("__")[0] for f in sc_files]
NAIVE_ = {}
for mk in MKS:
    p = os.path.join(RES, "llm_runs", f"{mk}__naive.jsonl")
    recs = [json.loads(l) for l in open(p) if l.strip()]
    ver = pd.Series([r.get("run_version") for r in recs]).value_counts().index[0]
    NAIVE_[mk] = {r["sample_id"]: r for r in recs if r.get("run_version") == ver}

rows, sweeps = [], []
for mk in MKS:
    sc = {}
    for line in open(os.path.join(RES, "sc_runs", f"{mk}__selfconsistency.jsonl")):
        if line.strip():
            r = json.loads(line); sc[r["sample_id"]] = r
    test_ids = [s for s in sc if SPLIT.get(s) == "test"]
    cal_ids = [s for s in sc if SPLIT.get(s) == "calib"]
    n_samples = sc[test_ids[0]]["n"]

    for dname in sorted({DS[s] for s in test_ids}) + ["ALL"]:
        ids = [s for s in test_ids if dname in (DS[s], "ALL")] if dname != "ALL" else test_ids
        y = np.array([Y[s] for s in ids])
        maj, agr = zip(*(vote(sc[s]) for s in ids))
        maj_pred = np.array([YR[s] if m is None else int(m == "attack") for s, m in zip(ids, maj)])
        greedy = np.array([YR[s] if NAIVE_[mk][s]["llm_answer"] is None
                           else int(NAIVE_[mk][s]["llm_answer"] == "attack") for s in ids])
        rf = np.array([YR[s] for s in ids])
        agr = np.array(agr)
        wrong = (maj_pred != y).astype(int)
        rows.append({
            "Model": mk, "Dataset": dname, "n": len(ids), "Samples per flow": n_samples,
            "Reflex only (%)": pct(rf == y), "Greedy LLM (%)": pct(greedy == y),
            "Majority vote (%)": pct(maj_pred == y),
            "Mean agreement": round(float(agr.mean()), 3),
            "Unanimous flows (%)": round(100 * float((agr == 1.0).mean()), 1),
            "AUROC of disagreement for a wrong answer": auc_ci(wrong, 1 - agr),
            "Tokens per flow": round(float(np.mean([sc[s]["tokens"] for s in ids])), 1),
            "Latency per flow (s)": round(float(np.mean([sc[s]["latency"] + RLAT[DS[s]] for s in ids])), 2)})

    # fallback-to-reflex sweep over the agreement threshold
    for thr in sorted({round(k / n_samples, 3) for k in range(1, n_samples + 1)}) + [1.01]:
        def acc(ids):
            y = np.array([Y[s] for s in ids])
            pred = []
            for s in ids:
                lab, a = vote(sc[s])
                pred.append(YR[s] if (lab is None or a < thr) else int(lab == "attack"))
            return accuracy_score(y, pred), np.mean([vote(sc[s])[1] < thr for s in ids])
        a_cal, _ = acc(cal_ids) if cal_ids else (np.nan, np.nan)
        a_tst, fb = acc(test_ids)
        sweeps.append({"Model": mk, "Agreement threshold": thr,
                       "Calibration accuracy (%)": round(100 * a_cal, 1),
                       "Test accuracy (%)": round(100 * a_tst, 1),
                       "Falls back to reflex (%)": round(100 * fb, 1)})

T15 = pd.DataFrame(rows); T16 = pd.DataFrame(sweeps)
T15.to_csv(os.path.join(RES, "tables", "T15_self_consistency_detail.csv"), index=False)
T16.to_csv(os.path.join(RES, "tables", "T16_self_consistency_threshold_sweep.csv"), index=False)
print("[saved] tables/T15_self_consistency_detail.csv, tables/T16_self_consistency_threshold_sweep.csv")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
print("\n=== T15 self-consistency, per dataset and overall\n", T15.to_string(index=False))
print("\n=== T16 agreement-threshold sweep (fallback to the reflex below the threshold)\n", T16.to_string(index=False))